In [147]:
!pip -q install pandas numpy cryptography

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.


In [ ]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
json_paths

[]

In [ ]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "No all_data_*.json found in /content. Upload the exports."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce").astype("Int64")
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"], na_position="last").reset_index(drop=True)

telemetry.head(), telemetry.shape

(                     source_file script_name                  timestamp  \
 0  all_data_20250817_095650.json       model 2025-08-17 09:56:43.222236   
 1  all_data_20250817_095650.json      cookie 2025-08-17 09:56:43.749270   
 2  all_data_20250817_095650.json      client 2025-08-17 09:56:44.239768   
 3  all_data_20250817_095650.json       model 2025-08-17 09:56:45.642759   
 4  all_data_20250817_095650.json       model 2025-08-17 09:56:45.643461   
 
    cycle  classical_host  classical_mate  classical_shared  quantum_host  \
 0      1           0.714           0.733             0.709         0.376   
 1      1           0.562           0.710             0.603         0.272   
 2      1           0.468           0.714             0.349         0.288   
 3      1           0.714           0.733             0.709         0.376   
 4      2           0.714           0.733             0.709         0.376   
 
    quantum_mate  quantum_shared  
 0         0.328           0.240  
 1      

In [ ]:
import numpy as np
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend

def key_from_row(row, salt=b"hive-v1", info=b"sentiment-key"):
    # robust quantization: map floats -> int16 bytes deterministically
    vec = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=np.float64)
    if np.any(pd.isna(vec)):
        return None
    # clip to [-1,1] then scale
    vec = np.clip(vec, -1.0, 1.0)
    q = (vec * 32767.0).round().astype(np.int16)
    ikm = q.tobytes() + str(row.get("cycle")).encode()  # include cycle for uniqueness

    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

# attach keys for rows that have quantum values
telemetry["key32"] = telemetry.apply(key_from_row, axis=1)
telemetry[telemetry["key32"].notna()].head(5)


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,key32
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\x1cD\xd2\xb5\x9c\xf2M\xaa\xd6\xe3\x18uu\xc5...
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,b' B~Fk\xbb\xe5\xafo=\xb8\x1e\xb9\x04ipI0\xc1}...
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,b'\xf0\xe5k\x8c\xc3\xf2\x9c\xef\xe5\xdf\x82\x0...
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\x1cD\xd2\xb5\x9c\xf2M\xaa\xd6\xe3\x18uu\xc5...
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,b'_D\x90\xba3\x9e>\x04\xd3\xdfJ\xfc\xe7\x9b\x1...


In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os, base64

def encrypt_with_row(row, plaintext: bytes, aad: bytes = b"hive"):
    key = row["key32"]
    if key is None:
        raise ValueError("Row has no key")
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return {
        "nonce_b64": base64.b64encode(nonce).decode(),
        "ct_b64": base64.b64encode(ct).decode(),
        "cycle": int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        "timestamp": str(row["timestamp"]),
        "script_name": row["script_name"],
    }

def decrypt_with_row(row, payload, aad: bytes = b"hive"):
    key = row["key32"]
    aesgcm = AESGCM(key)
    nonce = base64.b64decode(payload["nonce_b64"])
    ct = base64.b64decode(payload["ct_b64"])
    return aesgcm.decrypt(nonce, ct, aad)

# pick a row that has a key
row = telemetry[telemetry["key32"].notna()].iloc[0]
payload = encrypt_with_row(row, b"hello hive: quantum-locked message")
payload


{'nonce_b64': '8FCcHhqt/FH/GaB/',
 'ct_b64': 'YqpaaYwTZP2QaT6x58GyP7/M+pj7MK09ROvZAY3MyxcKHntRqrSNwfDMm5Kz73H9INM=',
 'cycle': 1,
 'timestamp': '2025-08-17 09:56:43.222236',
 'script_name': 'model'}

In [ ]:
import sqlite3, hashlib, json
from pathlib import Path

db_path = "/content/collected_data.db"
assert Path(db_path).exists(), "Upload collected_data.db into /content first"

con = sqlite3.connect(db_path)
cur = con.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha256 TEXT
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_messages (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT,
  cycle INTEGER,
  script_name TEXT,
  aad TEXT,
  nonce_b64 TEXT,
  ct_b64 TEXT,
  key_sha256 TEXT,
  meta_json TEXT
)
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_tel_cycle ON hive_telemetry(cycle)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_msg_cycle ON hive_messages(cycle)")
con.commit()

print("DB ready:", db_path)


DB ready: /content/collected_data.db


In [ ]:
import pandas as pd
import hashlib # Added missing import

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

# Ensure the database schema is up-to-date for this cell's operation
# Drop the table if it exists to allow schema re-creation with the key_sha256 column
cur.execute("DROP TABLE IF EXISTS hive_telemetry")
cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha256 TEXT
)
""")
con.commit()

tel = telemetry.copy()
tel = tel[tel["key32"].notna()].copy()
tel["key_sha256"] = tel["key32"].apply(lambda k: sha256_hex(k))

# write to db (append)
cols = [
    "timestamp","cycle","script_name","source_file",
    "classical_host","classical_mate","classical_shared",
    "quantum_host","quantum_mate","quantum_shared",
    "key_sha256"
]
tel_to_write = tel[cols].copy()
tel_to_write["timestamp"] = tel_to_write["timestamp"].astype(str)

tel_to_write.to_sql("hive_telemetry", con, if_exists="append", index=False)

con.commit()
print("Inserted telemetry rows:", len(tel_to_write))

Inserted telemetry rows: 49


In [ ]:
pd.read_sql_query("SELECT cycle, script_name, timestamp, key_sha256 FROM hive_telemetry ORDER BY id DESC LIMIT 10", con)


,cycle,script_name,timestamp,key_sha256
0,1,blockheart,2025-08-17 10:00:17.212964,f9d1ecf7b4dd60af071665f49a5768740960141d7a665b...
1,1,blockheart,2025-08-17 10:00:17.127296,f9d1ecf7b4dd60af071665f49a5768740960141d7a665b...
2,1,model,2025-08-17 10:00:16.397031,ca1a6f12aad5c1e27eb6e48b332f6e0b58eaa5026d4c36...
3,1,brian,2025-08-17 10:00:15.636896,e6fbbb78e882abc6fb1e3d5c76dcc5c25ab9d007663439...
4,4,cookie,2025-08-17 09:56:49.464948,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
5,4,cookie,2025-08-17 09:56:49.464790,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
6,4,cookie,2025-08-17 09:56:49.384530,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
7,4,cookie,2025-08-17 09:56:49.384412,ecbd04bcdc429120a2133726b1017082597920a3454627...
8,4,cookie,2025-08-17 09:56:49.384299,ecbd04bcdc429120a2133726b1017082597920a3454627...
9,4,cookie,2025-08-17 09:56:49.384174,ecbd04bcdc429120a2133726b1017082597920a3454627...


In [ ]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    # The error 'key32_cirq' indicates this column doesn't exist yet.
    # For robust execution, we will attempt to get 'key32_cirq' or 'key32_simple'.
    # If neither is found, it falls back to 'key32' (which is expected to exist from previous cells).
    # For the intended full functionality of 'key32_simple' and 'key32_cirq', ensure cell `Rz-2zsCoNJuD` is executed first.
    if key_kind == "cirq":
        key32 = row.get("key32_cirq", row.get("key32"))
    else:
        key32 = row.get("key32_simple", row.get("key32"))

    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError(f"Row has no valid key for kind={key_kind} or fallback 'key32'.")

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

# pick a row (you can choose a specific cycle later)
# Modified to use 'key32' for row selection as it is known to exist from a previously executed cell.
# For full intended functionality with 'key32_cirq', ensure cell `Rz-2zsCoNJuD` is executed first.
row = telemetry[telemetry["key32"].notna()].iloc[0]
payload = store_message(row, b"hello hive: stored in sqlite", key_kind="simple") # Adjusted key_kind for consistent fallback
payload

/tmp/ipython-input-4224040191.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('QfBIVd2CjyeV+u+d',
 'FUG7+g7h3NSWt3wHKHsKGerIFY1tVJdwtSPjc/jEdRwT2dRwc7sFczI3Zr4=',
 '34a65f0ff239b21a299084769984c3d06cf3f6b068b4437254b3c3375970b67c')

In [ ]:
pd.read_sql_query("SELECT sql FROM sqlite_master WHERE type='table' AND name='hive_messages'", con)

,sql
0,CREATE TABLE hive_messages (\n id INTEGER PRI...


In [ ]:
pd.read_sql_query("SELECT id, created_at, cycle, script_name, aad, key_sha256 FROM hive_messages ORDER BY id DESC LIMIT 5", con)

,id,created_at,cycle,script_name,aad,key_sha256


In [ ]:
def load_latest_message():
    df = pd.read_sql_query("SELECT * FROM hive_messages ORDER BY id DESC LIMIT 1", con)
    return df.iloc[0].to_dict()

msg = load_latest_message()
msg


{'id': 1,
 'created_at': '2026-01-26T11:34:19Z',
 'timestamp': '2025-08-17 09:56:43.222236',
 'cycle': 1,
 'script_name': 'model',
 'aad': 'hive',
 'nonce_b64': 'QfBIVd2CjyeV+u+d',
 'ct_b64': 'FUG7+g7h3NSWt3wHKHsKGerIFY1tVJdwtSPjc/jEdRwT2dRwc7sFczI3Zr4=',
 'key_sha': '34a65f0ff239b21a299084769984c3d06cf3f6b068b4437254b3c3375970b67c',
 'key_kind': 'simple',
 'meta_json': '{"source_file": "all_data_20250817_095650.json", "ts": "2025-08-17 09:56:43.222236", "classical": {"host": 0.714, "mate": 0.733, "shared": 0.709}, "quantum": {"host": 0.376, "mate": 0.328, "shared": 0.24}}'}

In [ ]:
cycle = msg["cycle"]
script = msg["script_name"]

candidate = telemetry[
    (telemetry["cycle"] == cycle) &
    (telemetry["script_name"] == script) &
    (telemetry["key32"].notna())
].iloc[0]

pt = decrypt_with_row(candidate, msg, aad=msg["aad"].encode())
pt


b'hello hive: stored in sqlite'

In [ ]:
!pip -q install cirq


In [ ]:
import cirq
import numpy as np

def circuit_projection_bits(row, n_qubits=6, reps=256):
    # map floats [-1,1] -> angles
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    # simple “sentiment embedding” across qubits
    for i,q in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(q))
        c.append(cirq.rz(a/2)(q))

    # light entanglement
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))

    sim = cirq.Simulator()
    res = sim.run(c, repetitions=reps)
    bits = res.measurements["m"]  # shape (reps, n_qubits)
    # compress to bytes deterministically
    packed = np.packbits(bits.astype(np.uint8), axis=1).tobytes()
    return packed  # bytes

def key_from_row_cirq(row, salt=b"hive-v1", info=b"cirq-proj"):
    ikm = circuit_projection_bits(row) + str(row.get("cycle")).encode()
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

# Example: generate cirq-based key
row = telemetry[telemetry["key32"].notna()].iloc[0]
k_cirq = key_from_row_cirq(row)
hashlib.sha256(k_cirq).hexdigest()[:16]


'4914bb7d608b62e0'

In [ ]:
!pip -q install pandas numpy cryptography scikit-learn cirq


In [ ]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "No all_data_*.json found in /content. Upload the exports."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce").astype("Int64")
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"], na_position="last").reset_index(drop=True)

telemetry.head(), telemetry.shape


(                     source_file script_name                  timestamp  \
 0  all_data_20250817_095650.json       model 2025-08-17 09:56:43.222236   
 1  all_data_20250817_095650.json      cookie 2025-08-17 09:56:43.749270   
 2  all_data_20250817_095650.json      client 2025-08-17 09:56:44.239768   
 3  all_data_20250817_095650.json       model 2025-08-17 09:56:45.642759   
 4  all_data_20250817_095650.json       model 2025-08-17 09:56:45.643461   
 
    cycle  classical_host  classical_mate  classical_shared  quantum_host  \
 0      1           0.714           0.733             0.709         0.376   
 1      1           0.562           0.710             0.603         0.272   
 2      1           0.468           0.714             0.349         0.288   
 3      1           0.714           0.733             0.709         0.376   
 4      2           0.714           0.733             0.709         0.376   
 
    quantum_mate  quantum_shared  
 0         0.328           0.240  
 1      

In [ ]:
import numpy as np
import hashlib
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend
import cirq

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def hkdf32(ikm: bytes, salt=b"hive-v1", info=b"sentiment-key") -> bytes:
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

def key_from_row_simple(row, salt=b"hive-v1", info=b"simple-v1"):
    vec = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=np.float64)
    if np.any(pd.isna(vec)):
        return None
    vec = np.clip(vec, -1.0, 1.0)
    q = (vec * 32767.0).round().astype(np.int16)
    ikm = q.tobytes() + str(int(row["cycle"]) if pd.notna(row["cycle"]) else -1).encode()
    return hkdf32(ikm, salt=salt, info=info)

def circuit_projection_bytes(row, n_qubits=6, reps=256):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    if np.any(pd.isna(v)):
        return None
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i,qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))

    sim = cirq.Simulator()
    res = sim.run(c, repetitions=reps)
    bits = res.measurements["m"].astype(np.uint8)  # (reps, n_qubits)
    packed = np.packbits(bits, axis=1).tobytes()
    return packed

def key_from_row_cirq(row, salt=b"hive-v1", info=b"cirq-proj-v1"):
    packed = circuit_projection_bytes(row)
    if packed is None:
        return None
    ikm = packed + str(int(row["cycle"]) if pd.notna(row["cycle"]) else -1).encode()
    return hkdf32(ikm, salt=salt, info=info)

telemetry["key32_simple"] = telemetry.apply(key_from_row_simple, axis=1)
telemetry["key32_cirq"]   = telemetry.apply(key_from_row_cirq, axis=1)

telemetry["key_sha_simple"] = telemetry["key32_simple"].apply(lambda k: sha256_hex(k) if isinstance(k,(bytes,bytearray)) else None)
telemetry["key_sha_cirq"]   = telemetry["key32_cirq"].apply(lambda k: sha256_hex(k) if isinstance(k,(bytes,bytearray)) else None)

telemetry[telemetry["key_sha_cirq"].notna()].head()


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,key32_simple,key32_cirq,key_sha_simple,key_sha_cirq
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\xb93\x9a\x08\x91\x1d\x13\xb64\')\xf1\x18\xf...,b'\x13u\x86\x92\x90\xc6\xfd\xdc^}I0\xc1\x1ck\x...,4bc5209475d2bfdd7c11f99841e59755ed1624bbcd37e6...,9c1600b4788f7cec7753641fb0c5ea700fd6fa40a37fd6...
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,b'\xb6\x07.g\xa4\xf9L\xdb\x1f:\x04\x114\xe5\x1...,b'\xf2\xa3\xfb=\xf2\x05\xf6\x1b\x1a\xd7\xf4\x1...,10e20666fd8fd70b1d1bd7baf01ecc3cdf86b3162b4edb...,fe15116f2c184083a38482a19ff9206958dba9df1504f0...
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,b'~\xaaG=\rUU\t\x8e\x057_\xc3\xe2\xebNQd3t\xfc...,"b""\xcdR\xcf\xecK\x9d_\x08\xee\xa0KH\x89\x1dz>\...",dd201754f43ceff13d37ac555b7f55838320978487b434...,0642ccce2ec6813f5dc37cd9a0ea697f7949989e954359...
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\xb93\x9a\x08\x91\x1d\x13\xb64\')\xf1\x18\xf...,b'(Zy\xce\xdf\xa0\xdb\x0c\tU\xd5\xf9Z\xe4\xd6\...,4bc5209475d2bfdd7c11f99841e59755ed1624bbcd37e6...,e717e3d29c507dd181f3d9c1aa61e7e58ffd4d71990b2b...
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,b'`d\xaa\xb0\xb0\x9d\xa5\xb0\xcd\xf3sjI\xc6r\x...,b'\xfb\xc9\xc7\xb6\x01;P[\x8a\x87kc\xd2\x8d\xc...,b66d4d66274f41dced77dd092582550091072aeb3ee84c...,7c51ca625c6c5732e33062f9ba28da753f1cd63a20e414...


In [ ]:
import sqlite3
from pathlib import Path

db_path = "/content/collected_data.db"
assert Path(db_path).exists(), "Upload collected_data.db into /content first"

con = sqlite3.connect(db_path)
cur = con.cursor()

# Drop the tables if they exist to ensure schema update
cur.execute("DROP TABLE IF EXISTS hive_telemetry")
cur.execute("DROP TABLE IF EXISTS hive_messages")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  timestamp TEXT NOT NULL,
  cycle INTEGER,
  script_name TEXT NOT NULL,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha_simple TEXT,
  key_sha_cirq TEXT,
  PRIMARY KEY (timestamp, script_name, cycle)
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_messages (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT NOT NULL,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  aad TEXT,
  nonce_b64 TEXT,
  ct_b64 TEXT,
  key_sha TEXT,
  key_kind TEXT,
  meta_json TEXT
)
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_tel_cycle ON hive_telemetry(cycle)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_msg_cycle ON hive_messages(cycle)")
con.commit()

print("DB schema ready.")

DB schema ready.


In [ ]:
tel = telemetry.copy()
tel = tel[tel["timestamp"].notna() & tel["script_name"].notna()].copy()

def to_py(v):
    if pd.isna(v): return None
    if isinstance(v, (pd.Timestamp,)): return v.isoformat()
    if isinstance(v, (pd._libs.missing.NAType,)): return None
    return v

rows = []
for _, r in tel.iterrows():
    rows.append((
        to_py(r["timestamp"]),
        int(r["cycle"]) if pd.notna(r["cycle"]) else None,
        str(r["script_name"]),
        str(r["source_file"]) if pd.notna(r["source_file"]) else None,
        to_py(r["classical_host"]), to_py(r["classical_mate"]), to_py(r["classical_shared"]),
        to_py(r["quantum_host"]),   to_py(r["quantum_mate"]),   to_py(r["quantum_shared"]),
        r["key_sha_simple"],
        r["key_sha_cirq"],
    ))

cur.executemany("""
INSERT INTO hive_telemetry (
  timestamp, cycle, script_name, source_file,
  classical_host, classical_mate, classical_shared,
  quantum_host, quantum_mate, quantum_shared,
  key_sha_simple, key_sha_cirq
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(timestamp, script_name, cycle) DO UPDATE SET
  source_file=excluded.source_file,
  classical_host=excluded.classical_host,
  classical_mate=excluded.classical_mate,
  classical_shared=excluded.classical_shared,
  quantum_host=excluded.quantum_host,
  quantum_mate=excluded.quantum_mate,
  quantum_shared=excluded.quantum_shared,
  key_sha_simple=excluded.key_sha_simple,
  key_sha_cirq=excluded.key_sha_cirq
""", rows)

con.commit()
print("Upserted telemetry rows:", len(rows))

Upserted telemetry rows: 49


In [ ]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    key32 = row["key32_cirq"] if key_kind == "cirq" else row["key32_simple"]
    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError("Row has no key for kind=" + key_kind)

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

row0 = telemetry[telemetry["key32_cirq"].notna()].iloc[0]
nonce_b64, ct_b64, key_sha = store_message(row0, b"hello hive: cirq-locked message", key_kind="cirq")
(key_sha, nonce_b64[:10], ct_b64[:10])

/tmp/ipython-input-2248142252.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('9c1600b4788f7cec7753641fb0c5ea700fd6fa40a37fd68d45630c3dfbc98f87',
 'lxJt2F+Huf',
 'TVj+3Q4Khu')

In [ ]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    key32 = row["key32_cirq"] if key_kind == "cirq" else row["key32_simple"]
    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError("Row has no key for kind=" + key_kind)

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

row0 = telemetry[telemetry["key32_cirq"].notna()].iloc[0]
nonce_b64, ct_b64, key_sha = store_message(row0, b"hello hive: cirq-locked message", key_kind="cirq")
(key_sha, nonce_b64[:10], ct_b64[:10])


/tmp/ipython-input-330299270.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('9c1600b4788f7cec7753641fb0c5ea700fd6fa40a37fd68d45630c3dfbc98f87',
 '3SRDuXWGHe',
 '0lUDUKszaL')

In [ ]:
import numpy as np

df = telemetry.copy()
df = df[df["timestamp"].notna()].copy()

# Basic features
df["hour"] = df["timestamp"].dt.hour.astype(float)
df["minute"] = df["timestamp"].dt.minute.astype(float)

# Target(s)
targets = ["quantum_host","quantum_mate","quantum_shared"]

# Keep rows where we have classical + quantum
needed = ["classical_host","classical_mate","classical_shared"] + targets
df = df.dropna(subset=needed + ["script_name"])

# One-hot encode script_name
X = df[["classical_host","classical_mate","classical_shared","hour","minute"]].copy()
X = pd.concat([X, pd.get_dummies(df["script_name"], prefix="script")], axis=1)

y_shared = df["quantum_shared"].astype(float).values
y_host   = df["quantum_host"].astype(float).values
y_mate   = df["quantum_mate"].astype(float).values

X.shape, df.shape


((49, 10), (49, 16))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y_shared, test_size=0.2, random_state=42)

reg = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))


MAE: 0.010541086652236511
R2 : -0.6333787491091651


In [ ]:
from sklearn.ensemble import IsolationForest

feat_cols = ["classical_host","classical_mate","classical_shared","quantum_host","quantum_mate","quantum_shared"]
A = df[feat_cols].astype(float).values

iso = IsolationForest(n_estimators=400, contamination=0.03, random_state=42)
scores = iso.fit_predict(A)  # -1 anomaly, +1 normal
df["anomaly"] = (scores == -1)

df[df["anomaly"]].head(20)[["timestamp","cycle","script_name"] + feat_cols]


,timestamp,cycle,script_name,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
32,2025-08-17 09:56:48.403926,3,client,0.786,0.485,0.733,0.456,0.344,0.344
36,2025-08-17 09:56:49.019539,4,model,0.633,0.588,0.293,0.424,0.416,0.208


In [ ]:
df.groupby("script_name")["anomaly"].mean().sort_values(ascending=False)


,anomaly
script_name,
client,0.090909
model,0.058824
blockheart,0.000000
brian,0.000000
cookie,0.000000


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

y_script = df["script_name"].values
Xc = df[feat_cols + ["hour","minute"]].astype(float)
Xc = (Xc - Xc.mean()) / (Xc.std() + 1e-9)  # standardize

# Check the distribution of classes in y_script
print("Script name counts:\n", df["script_name"].value_counts())

# Filter out script names with only one occurrence
script_counts = df["script_name"].value_counts()
scripts_to_keep = script_counts[script_counts > 1].index
df_filtered = df[df["script_name"].isin(scripts_to_keep)]

y_script_filtered = df_filtered["script_name"].values
Xc_filtered = Xc[df["script_name"].isin(scripts_to_keep)]

X_train, X_test, y_train, y_test = train_test_split(Xc_filtered, y_script_filtered, test_size=0.2, random_state=42, stratify=y_script_filtered)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Script name counts:
 script_name
cookie        18
model         17
client        11
blockheart     2
brian          1
Name: count, dtype: int64
Accuracy: 0.8
              precision    recall  f1-score   support

      client       0.00      0.00      0.00         2
      cookie       0.67      1.00      0.80         4
       model       1.00      1.00      1.00         4

    accuracy                           0.80        10
   macro avg       0.56      0.67      0.60        10
weighted avg       0.67      0.80      0.72        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

y_script = df["script_name"].values
Xc = df[feat_cols + ["hour","minute"]].astype(float)
Xc = (Xc - Xc.mean()) / (Xc.std() + 1e-9)  # standardize

# Filter out script names with only one occurrence to allow for stratified splitting
script_counts = df["script_name"].value_counts()
scripts_to_keep = script_counts[script_counts > 1].index
df_filtered = df[df["script_name"].isin(scripts_to_keep)]

y_script_filtered = df_filtered["script_name"].values
Xc_filtered = Xc[df["script_name"].isin(scripts_to_keep)]

X_train, X_test, y_train, y_test = train_test_split(Xc_filtered, y_script_filtered, test_size=0.2, random_state=42, stratify=y_script_filtered)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8
              precision    recall  f1-score   support

      client       0.00      0.00      0.00         2
      cookie       0.67      1.00      0.80         4
       model       1.00      1.00      1.00         4

    accuracy                           0.80        10
   macro avg       0.56      0.67      0.60        10
weighted avg       0.67      0.80      0.72        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("rows:", len(telemetry))
print("keys(simple):", telemetry["key32_simple"].notna().sum())
print("keys(cirq):", telemetry["key32_cirq"].notna().sum())
print("unique scripts:", telemetry["script_name"].nunique())
print("scripts:", sorted(telemetry["script_name"].dropna().unique())[:20])


rows: 49
keys(simple): 49
keys(cirq): 49
unique scripts: 5
scripts: ['blockheart', 'brian', 'client', 'cookie', 'model']


In [ ]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio

# Cirq + qsimcirq can be finicky; this usually works in Colab:
!pip -q install cirq qsimcirq

# Brian2
!pip -q install brian2


In [ ]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=["timestamp","script_name","cycle",
                                     "classical_host","classical_mate","classical_shared",
                                     "quantum_host","quantum_mate","quantum_shared"]).copy()
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)

print("rows:", len(telemetry), "scripts:", telemetry["script_name"].nunique())
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240


In [ ]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=["timestamp","script_name","cycle",
                                     "classical_host","classical_mate","classical_shared",
                                     "quantum_host","quantum_mate","quantum_shared"]).copy()
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)

print("rows:", len(telemetry), "scripts:", telemetry["script_name"].nunique())
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240


In [ ]:
import cirq

# Try qsimcirq (fast). Fallback to cirq.Simulator if not available.
try:
    import qsimcirq
    HAVE_QSIM = True
except Exception:
    HAVE_QSIM = False

def make_circuit_from_row(row, n_qubits=6):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i, qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))
    c.append(cirq.measure(*qs, key="m"))
    return c, qs

def sample_bits(circuit, reps=256):
    if HAVE_QSIM:
        sim = qsimcirq.QSimSimulator()
    else:
        sim = cirq.Simulator()
    res = sim.run(circuit, repetitions=reps)
    bits = res.measurements["m"].astype(np.uint8)  # (reps, n_qubits)
    return bits

def bits_to_hist(bits):
    # convert each measurement to integer 0..(2^n-1), then histogram
    reps, n = bits.shape
    vals = (bits * (2 ** np.arange(n)[None, :])).sum(axis=1)
    hist = np.bincount(vals, minlength=2**n).astype(np.float32)
    hist /= max(1, hist.sum())
    return hist

# quick test
r0 = telemetry.iloc[0].to_dict()
c0, _ = make_circuit_from_row(r0, n_qubits=6)
b0 = sample_bits(c0, reps=256)
f0 = bits_to_hist(b0)
print("feature dim:", f0.shape, "qsim:", HAVE_QSIM)


feature dim: (64,) qsim: True


In [ ]:
from tqdm import tqdm

N_QUBITS = 6
REPS = 256
FEAT_DIM = 2**N_QUBITS

features = np.zeros((len(telemetry), FEAT_DIM), dtype=np.float32)

for i in tqdm(range(len(telemetry))):
    row = telemetry.iloc[i].to_dict()
    c, _ = make_circuit_from_row(row, n_qubits=N_QUBITS)
    bits = sample_bits(c, reps=REPS)
    features[i] = bits_to_hist(bits)

print("features:", features.shape)


100%|██████████| 49/49 [00:00<00:00, 216.10it/s]

features: (49, 64)


In [ ]:
from collections import defaultdict

telemetry = telemetry.reset_index(drop=True)
telemetry["script_id"] = telemetry["script_name"].astype("category").cat.codes
n_scripts = telemetry["script_id"].nunique()

# group indices by script
by_script = defaultdict(list)
for idx, sid in enumerate(telemetry["script_id"].values):
    by_script[int(sid)].append(idx)

SEQ_LEN = 5  # Reduced window length to allow sequence creation

X_seq = []
Y_next = []
S_seq = []

targets = telemetry[["quantum_host","quantum_mate","quantum_shared"]].values.astype(np.float32)

for sid, idxs in by_script.items():
    # ensure time order already sorted; idxs are in sorted order due to global sort by timestamp+script
    for j in range(0, len(idxs) - SEQ_LEN - 1):
        win = idxs[j:j+SEQ_LEN]
        nxt = idxs[j+SEQ_LEN]
        X_seq.append(features[win])             # (SEQ_LEN, FEAT_DIM)
        Y_next.append(targets[nxt])             # (3,)
        S_seq.append(sid)                       # script id (observer identity)

X_seq = np.stack(X_seq).astype(np.float32)
Y_next = np.stack(Y_next).astype(np.float32)
S_seq = np.array(S_seq, dtype=np.int64)

print("dataset:", X_seq.shape, Y_next.shape, "scripts:", n_scripts)

dataset: (28, 5, 64) (28, 3) scripts: 5


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class HiveDataset(Dataset):
    def __init__(self, X, Y, S):
        self.X = torch.from_numpy(X)   # (N, T, D)
        self.Y = torch.from_numpy(Y)   # (N, 3)
        self.S = torch.from_numpy(S)   # (N,)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i], self.S[i]

class TransformerPredictor(nn.Module):
    def __init__(self, feat_dim, n_scripts, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)
        self.in_proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, 3)
        )

    def forward(self, x, sid):
        # x: (B,T,D)
        h = self.in_proj(x)
        h = h + self.script_emb(sid).unsqueeze(1)  # add observer embedding
        h = self.encoder(h)
        last = h[:, -1, :]
        return self.out(last)

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerPredictor(FEAT_DIM, n_scripts).to(torch_device)
model

TransformerPredictor(
  (script_emb): Embedding(5, 256)
  (in_proj): Linear(in_features=64, out_features=256, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): GELU(approximate=

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X_seq))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

ds_train = HiveDataset(X_seq[train_idx], Y_next[train_idx], S_seq[train_idx])
ds_val   = HiveDataset(X_seq[val_idx],   Y_next[val_idx],   S_seq[val_idx])

dl_train = DataLoader(ds_train, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.SmoothL1Loss()

def eval_loss():
    model.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        for x,y,sid in dl_val:
            x,y,sid = x.to(torch_device), y.to(torch_device), sid.to(torch_device)
            pred = model(x, sid)
            loss = loss_fn(pred, y)
            tot += float(loss) * x.size(0)
            n += x.size(0)
    return tot/n

for epoch in range(1, 1000):
    model.train()
    tot, n = 0.0, 0
    for x,y,sid in dl_train:
        x,y,sid = x.to(torch_device), y.to(torch_device), sid.to(torch_device)
        opt.zero_grad(set_to_none=True)
        pred = model(x, sid)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += float(loss) * x.size(0)
        n += x.size(0)

    print(f"epoch {epoch:02d} train {tot/n:.5f}  val {eval_loss():.5f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipython-input-3226539440.py:38: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * x.size(0)


epoch 01 train 0.08335  val 0.01182
epoch 02 train 0.01106  val 0.01055
epoch 03 train 0.00972  val 0.00431
epoch 04 train 0.00428  val 0.00653
epoch 05 train 0.00745  val 0.00557
epoch 06 train 0.00469  val 0.00394
epoch 07 train 0.00299  val 0.00339
epoch 08 train 0.00288  val 0.00338
epoch 09 train 0.00291  val 0.00312
epoch 10 train 0.00353  val 0.00234
epoch 11 train 0.00231  val 0.00168
epoch 12 train 0.00177  val 0.00147
epoch 13 train 0.00145  val 0.00165
epoch 14 train 0.00177  val 0.00185
epoch 15 train 0.00230  val 0.00183
epoch 16 train 0.00209  val 0.00168
epoch 17 train 0.00195  val 0.00154
epoch 18 train 0.00126  val 0.00155
epoch 19 train 0.00117  val 0.00168
epoch 20 train 0.00134  val 0.00181
epoch 21 train 0.00177  val 0.00178
epoch 22 train 0.00155  val 0.00166
epoch 23 train 0.00156  val 0.00154
epoch 24 train 0.00134  val 0.00145
epoch 25 train 0.00129  val 0.00140
epoch 26 train 0.00124  val 0.00137
epoch 27 train 0.00121  val 0.00132
epoch 28 train 0.00102  val 

In [ ]:
from brian2 import *

def brian2_decode(pred_vec, duration_ms=200):
    # pred_vec: (3,) floats in [-1,1] roughly (your targets are around 0..1, but we clip)
    v = np.array(pred_vec, dtype=float)
    v = np.clip(v, -1.0, 1.0)

    start_scope()
    defaultclock.dt = 0.1*ms

    N = 64
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''
    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')

    # map vector to current profile
    # (simple: base + weighted components)
    base = 0.6
    weights = np.linspace(0.5, 1.5, N)
    drive = base + weights*(0.2*v[0] + 0.2*v[1] + 0.2*v[2])
    G.I = drive

    M = SpikeMonitor(G)
    run(duration_ms*ms)

    # return spike count and rate estimate
    count = M.count[:]  # spikes per neuron
    rate_hz = count.mean() / (duration_ms/1000.0)
    return float(rate_hz), count

# demo: run decoder on a sample
model.eval()
x,y,sid = ds_val[0]
with torch.no_grad():
    pred = model(x.unsqueeze(0).to(torch_device), torch.tensor([int(sid)]).to(torch_device)).cpu().numpy()[0]

rate_hz, counts = brian2_decode(pred, duration_ms=200)
rate_hz, pred

WARNING    'v' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value array([0.32669815, 0.36610591, 0.25764552]). The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


(0.0, array([0.32669815, 0.3661059 , 0.25764552], dtype=float32))

In [ ]:
telemetry.groupby("script_name")["cycle"].count().sort_values(ascending=False)


,cycle
script_name,
cookie,18
model,17
client,11
blockheart,2
brian,1


In [ ]:
!pip -q install torch torchvision torchaudio transformers sentence-transformers accelerate \
  opencv-python pillow librosa ffmpeg-python pandas numpy scikit-learn

# optional (if you want CLIP):
!pip -q install ftfy regex tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00


In [ ]:
{"timestamp":"2025-08-17T09:56:43.222236","script":"model","cycle":1,
 "text":"operator note ...", "image_path":".../frame_0001.jpg",
 "audio_path":".../clip.wav", "video_path":".../clip.mp4"}


{'timestamp': '2025-08-17T09:56:43.222236',
 'script': 'model',
 'cycle': 1,
 'text': 'operator note ...',
 'image_path': '.../frame_0001.jpg',
 'audio_path': '.../clip.wav',
 'video_path': '.../clip.mp4'}

In [ ]:
from sentence_transformers import SentenceTransformer
text_teacher = SentenceTransformer("all-MiniLM-L6-v2")  # small + good
text_teacher.eval()


WARNING    /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
 [py.warnings]
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [ ]:
from transformers import WhisperProcessor, WhisperModel
whisper_model = WhisperModel.from_pretrained("openai/whisper-small")  # encoder+decoder, we use encoder
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model.eval()


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 768)
    (layers): ModuleList(
      (0-11): 12 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      

In [ ]:
from transformers import WhisperProcessor, WhisperModel
whisper_model = WhisperModel.from_pretrained("openai/whisper-small")  # encoder+decoder, we use encoder
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model.eval()


WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 768)
    (layers): ModuleList(
      (0-11): 12 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      

In [ ]:
import numpy as np
from PIL import Image
import librosa
import cv2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = clip_model.to(DEVICE)
whisper_model = whisper_model.to(DEVICE)

@torch.no_grad()
def embed_text(t: str):
    if not t:
        return None
    v = text_teacher.encode([t], normalize_embeddings=True)[0].astype(np.float32)
    return v  # (d,)

@torch.no_grad()
def embed_image(path: str):
    if not path:
        return None
    img = Image.open(path).convert("RGB")
    inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats[0].cpu().numpy().astype(np.float32)  # (512,)

@torch.no_grad()
def embed_audio(path: str, sr=16000, max_sec=20):
    if not path:
        return None
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = wav[: sr*max_sec]
    inputs = whisper_proc(wav, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)  # (1, 80, frames)
    enc = whisper_model.encoder(input_features).last_hidden_state  # (1, T, d)
    # pool
    v = enc.mean(dim=1)[0]
    v = torch.nn.functional.normalize(v, dim=-1)
    return v.cpu().numpy().astype(np.float32)  # (d,)

@torch.no_grad()
def embed_video(path: str, n_frames=8):
    if not path:
        return None
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(0, frame_count-1), n_frames).astype(int)

    embs = []
    cur = 0
    want = set(idxs.tolist())
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame)
            inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
            feats = clip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            embs.append(feats[0].cpu().numpy())
        i += 1
    cap.release()
    if not embs:
        return None
    v = np.mean(np.stack(embs).astype(np.float32), axis=0)
    v = v / (np.linalg.norm(v) + 1e-9)
    return v.astype(np.float32)  # (512,)


In [ ]:
import torch
import torch.nn as nn

class MultiModalStudent(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=4,
                 d_text=384, d_img=512, d_vid=512, d_aud=768, d_telem=64,
                 n_scripts=16):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)

        self.p_text  = nn.Linear(d_text, d_model)
        self.p_img   = nn.Linear(d_img, d_model)
        self.p_vid   = nn.Linear(d_vid, d_model)
        self.p_aud   = nn.Linear(d_aud, d_model)
        self.p_tel   = nn.Linear(d_telem, d_model)

        # token type embeddings: 0=TEL,1=TXT,2=IMG,3=AUD,4=VID
        self.type_emb = nn.Embedding(5, d_model)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=0.1, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=num_layers)

        # heads
        self.next_q = nn.Linear(d_model, 3)  # predict next (quantum_host,mate,shared)

        # distill heads (optional): match teacher embeddings (project back)
        self.dist_text = nn.Linear(d_model, d_text)
        self.dist_img  = nn.Linear(d_model, d_img)
        self.dist_aud  = nn.Linear(d_model, d_aud)
        self.dist_vid  = nn.Linear(d_model, d_vid)

    def forward(self, tel_seq, txt_seq, img_seq, aud_seq, vid_seq, mask_seq, script_id):
        """
        Inputs are sequences over time T:
          tel_seq: (B,T,d_telem)
          txt_seq/img_seq/aud_seq/vid_seq: (B,T,d_mod) or zeros if missing
          mask_seq: (B,T,5) 1 if present else 0
        We expand each time step into up to 5 tokens and flatten to (B, T*5, d_model).
        """
        B,T,_ = tel_seq.shape

        # project each modality
        tel = self.p_tel(tel_seq) + self.type_emb(torch.zeros((B,T),dtype=torch.long,device=tel_seq.device))
        txt = self.p_text(txt_seq) + self.type_emb(torch.ones((B,T),dtype=torch.long,device=tel_seq.device))
        img = self.p_img(img_seq)  + self.type_emb(torch.full((B,T),2,dtype=torch.long,device=tel_seq.device))
        aud = self.p_aud(aud_seq)  + self.type_emb(torch.full((B,T),3,dtype=torch.long,device=tel_seq.device))
        vid = self.p_vid(vid_seq)  + self.type_emb(torch.full((B,T),4,dtype=torch.long,device=tel_seq.device))

        # add observer/script embedding to all tokens
        s = self.script_emb(script_id).unsqueeze(1).unsqueeze(1)  # (B,1,1,d)
        tel,txt,img,aud,vid = tel+s,txt+s,img+s,aud+s,vid+s

        # stack tokens per time: (B,T,5,d) -> (B,T*5,d)
        tokens = torch.stack([tel,txt,img,aud,vid], dim=2)
        tokens = tokens.reshape(B, T*5, -1)

        # attention mask: True means "ignore"
        present = mask_seq.reshape(B, T*5)  # 1 present, 0 absent
        src_key_padding_mask = (present == 0)

        h = self.tr(tokens, src_key_padding_mask=src_key_padding_mask)

        # use the last TELEMETRY token at final timestep as summary (index = (T-1)*5 + 0)
        idx = (T-1)*5 + 0
        summary = h[:, idx, :]

        next_q = self.next_q(summary)

        # distill: predict teacher embeddings for *this timestep summary* (you can also do per-modality token)
        return next_q, {
            "text": self.dist_text(summary),
            "img":  self.dist_img(summary),
            "aud":  self.dist_aud(summary),
            "vid":  self.dist_vid(summary),
        }


In [ ]:
def cosine_loss(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return 1.0 - (a*b).sum(dim=-1).mean()


In [ ]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio transformers sentence-transformers \
  opencv-python pillow librosa ffmpeg-python matplotlib tqdm

# Cirq + qsimcirq + Brian2
!pip -q install cirq qsimcirq brian2


In [ ]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=[
    "timestamp","script_name","cycle",
    "classical_host","classical_mate","classical_shared",
    "quantum_host","quantum_mate","quantum_shared"
]).copy()

telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)
telemetry["script_id"] = telemetry["script_name"].astype("category").cat.codes
n_scripts = telemetry["script_id"].nunique()

print("rows:", len(telemetry), "scripts:", n_scripts)
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,script_id
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,4
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,3
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,2
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,4
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,4


In [ ]:
import cirq

try:
    import qsimcirq
    HAVE_QSIM = True
except Exception:
    HAVE_QSIM = False

def make_circuit_from_row(row, n_qubits=6):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i, qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))
    return c

def sample_bits(circuit, reps=256):
    sim = qsimcirq.QSimSimulator() if HAVE_QSIM else cirq.Simulator()
    res = sim.run(circuit, repetitions=reps)
    return res.measurements["m"].astype(np.uint8)

def bits_to_hist(bits):
    reps, n = bits.shape
    vals = (bits * (2 ** np.arange(n)[None, :])).sum(axis=1)
    hist = np.bincount(vals, minlength=2**n).astype(np.float32)
    hist /= max(1, hist.sum())
    return hist

from tqdm import tqdm

N_QUBITS = 6
REPS = 256
FEAT_DIM = 2**N_QUBITS

telemetry_feats = np.zeros((len(telemetry), FEAT_DIM), dtype=np.float32)
for i in tqdm(range(len(telemetry))):
    row = telemetry.iloc[i].to_dict()
    c = make_circuit_from_row(row, n_qubits=N_QUBITS)
    bits = sample_bits(c, reps=REPS)
    telemetry_feats[i] = bits_to_hist(bits)

print("telemetry_feats:", telemetry_feats.shape, "qsim:", HAVE_QSIM)


100%|██████████| 49/49 [00:00<00:00, 197.77it/s]

telemetry_feats: (49, 64) qsim: True


In [ ]:
import os, json, math
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import wave
import cv2

outdir = Path("/content/synth")
(outdir/"images").mkdir(parents=True, exist_ok=True)
(outdir/"audio").mkdir(parents=True, exist_ok=True)
(outdir/"video").mkdir(parents=True, exist_ok=True)
(outdir/"text").mkdir(parents=True, exist_ok=True)

def synth_text(row):
    # short “operator note” that encodes state
    ch, cm, cs = row["classical_host"], row["classical_mate"], row["classical_shared"]
    qh, qm, qs = row["quantum_host"], row["quantum_mate"], row["quantum_shared"]
    script = row["script_name"]
    cyc = int(row["cycle"])
    # simple narrative
    mood = "stable" if abs((qs - cs)) < 0.1 else ("drifting" if (qs < cs) else "amplifying")
    return (
        f"[{script}] cycle={cyc} mood={mood}. "
        f"classical(H={ch:.3f}, M={cm:.3f}, S={cs:.3f}) "
        f"quantum(H={qh:.3f}, M={qm:.3f}, S={qs:.3f})."
    )

def save_plot_image(row, path_png):
    ch, cm, cs = row["classical_host"], row["classical_mate"], row["classical_shared"]
    qh, qm, qs = row["quantum_host"], row["quantum_mate"], row["quantum_shared"]
    fig = plt.figure(figsize=(4, 3), dpi=120)
    ax = fig.add_subplot(111)
    ax.bar(["c_host","c_mate","c_shared","q_host","q_mate","q_shared"], [ch,cm,cs,qh,qm,qs])
    ax.set_ylim(0, 1)
    ax.set_title(f'{row["script_name"]} c{int(row["cycle"])}')
    fig.tight_layout()
    fig.savefig(path_png)
    plt.close(fig)

def save_audio_tone(row, path_wav, sr=16000, dur=1.0):
    # map Host/Mate/Shared to 3 tone freqs
    qh, qm, qs = float(row["quantum_host"]), float(row["quantum_mate"]), float(row["quantum_shared"])
    base = 220.0
    f1 = base * (1.0 + qh)
    f2 = base * (1.0 + qm) * 1.25
    f3 = base * (1.0 + qs) * 1.5
    t = np.linspace(0, dur, int(sr*dur), endpoint=False)
    sig = (0.33*np.sin(2*np.pi*f1*t) + 0.33*np.sin(2*np.pi*f2*t) + 0.33*np.sin(2*np.pi*f3*t))
    sig = sig / (np.max(np.abs(sig)) + 1e-9)
    pcm = (sig * 32767).astype(np.int16)

    with wave.open(str(path_wav), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())

def save_video_from_frames(row, path_mp4, n_frames=12, fps=12):
    # animate by slowly interpolating classical->quantum bars
    ch, cm, cs = float(row["classical_host"]), float(row["classical_mate"]), float(row["classical_shared"])
    qh, qm, qs = float(row["quantum_host"]), float(row["quantum_mate"]), float(row["quantum_shared"])

    W, H = 480, 360
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(str(path_mp4), fourcc, fps, (W, H))

    for k in range(n_frames):
        a = k/(n_frames-1)
        h = (1-a)*ch + a*qh
        m = (1-a)*cm + a*qm
        s = (1-a)*cs + a*qs

        # render simple bars with PIL
        img = Image.new("RGB", (W, H), (20, 20, 24))
        draw = ImageDraw.Draw(img)
        draw.text((12, 10), f'{row["script_name"]} c{int(row["cycle"])} a={a:.2f}', fill=(230,230,230))

        vals = [h, m, s]
        labels = ["host", "mate", "shared"]
        x0 = 60
        for i,(lab,val) in enumerate(zip(labels, vals)):
            x = x0 + i*120
            y_base = 320
            bar_h = int(240 * max(0.0, min(1.0, val)))
            draw.rectangle([x, y_base-bar_h, x+60, y_base], fill=(90, 170, 240))
            draw.text((x, y_base+8), lab, fill=(230,230,230))
            draw.text((x, y_base- bar_h - 18), f"{val:.2f}", fill=(230,230,230))

        frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        vw.write(frame)

    vw.release()

manifest_path = outdir/"manifest.jsonl"
with open(manifest_path, "w") as f:
    for i in tqdm(range(len(telemetry))):
        row = telemetry.iloc[i].to_dict()
        ts = telemetry.iloc[i]["timestamp"].isoformat()
        script = row["script_name"]
        cyc = int(row["cycle"])

        # file naming
        stem = f"{script}_c{cyc:06d}_{i:07d}"
        txt_path = outdir/"text"/f"{stem}.txt"
        img_path = outdir/"images"/f"{stem}.png"
        wav_path = outdir/"audio"/f"{stem}.wav"
        mp4_path = outdir/"video"/f"{stem}.mp4"

        txt = synth_text(row)
        txt_path.write_text(txt, encoding="utf-8")
        save_plot_image(row, img_path)
        save_audio_tone(row, wav_path)
        save_video_from_frames(row, mp4_path)

        rec = {
            "i": i,
            "timestamp": ts,
            "script_name": script,
            "script_id": int(telemetry.iloc[i]["script_id"]),
            "cycle": cyc,
            "text": txt,
            "image_path": str(img_path),
            "audio_path": str(wav_path),
            "video_path": str(mp4_path)
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

100%|██████████| 49/49 [00:15<00:00,  3.08it/s]


In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel, WhisperProcessor, WhisperModel
import librosa
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

text_teacher = SentenceTransformer("all-MiniLM-L6-v2")  # 384-d
text_teacher.eval()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

whisper_model = WhisperModel.from_pretrained("openai/whisper-small").to(DEVICE).eval()
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")


In [ ]:
import numpy as np
import cv2
from tqdm import tqdm

def embed_text(t: str):
    v = text_teacher.encode([t], normalize_embeddings=True)[0].astype(np.float32)
    return v  # (384,)

@torch.no_grad()
def embed_image(path: str):
    img = Image.open(path).convert("RGB")
    inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats[0].cpu().numpy().astype(np.float32)  # (512,)

@torch.no_grad()
def embed_audio(path: str, sr=16000, max_sec=2):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = wav[: sr*max_sec]
    inputs = whisper_proc(wav, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)
    enc = whisper_model.encoder(input_features).last_hidden_state  # (1,T,768)
    v = enc.mean(dim=1)[0]
    v = torch.nn.functional.normalize(v, dim=-1)
    return v.cpu().numpy().astype(np.float32)  # (768,)

@torch.no_grad()
def embed_video(path: str, n_frames=8):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return np.zeros((512,), dtype=np.float32)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(0, frame_count-1), n_frames).astype(int)
    want = set(idxs.tolist())

    embs = []
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame)
            inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
            feats = clip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            embs.append(feats[0].cpu().numpy())
        i += 1
    cap.release()
    if not embs:
        return np.zeros((512,), dtype=np.float32)
    v = np.mean(np.stack(embs).astype(np.float32), axis=0)
    v = v / (np.linalg.norm(v) + 1e-9)
    return v.astype(np.float32)

# Read manifest
records = [json.loads(line) for line in open(manifest_path, "r", encoding="utf-8")]

D_TEXT, D_IMG, D_AUD, D_VID = 384, 512, 768, 512
E_text = np.zeros((len(records), D_TEXT), dtype=np.float32)
E_img  = np.zeros((len(records), D_IMG),  dtype=np.float32)
E_aud  = np.zeros((len(records), D_AUD),  dtype=np.float32)
E_vid  = np.zeros((len(records), D_VID),  dtype=np.float32)

for r in tqdm(records):
    i = r["i"]
    E_text[i] = embed_text(r["text"])
    E_img[i]  = embed_image(r["image_path"])
    E_aud[i]  = embed_audio(r["audio_path"])
    E_vid[i]  = embed_video(r["video_path"])

np.save(outdir/"E_text.npy", E_text)
np.save(outdir/"E_img.npy",  E_img)
np.save(outdir/"E_aud.npy",  E_aud)
np.save(outdir/"E_vid.npy",  E_vid)

print("cached embeddings saved in", outdir)


100%|██████████| 49/49 [04:21<00:00,  5.35s/it]

cached embeddings saved in /content/synth


In [ ]:
from collections import defaultdict

targets = telemetry[["quantum_host","quantum_mate","quantum_shared"]].values.astype(np.float32)
script_ids = telemetry["script_id"].values.astype(np.int64)

# modality embeddings
E_text = np.load(outdir/"E_text.npy").astype(np.float32)
E_img  = np.load(outdir/"E_img.npy").astype(np.float32)
E_aud  = np.load(outdir/"E_aud.npy").astype(np.float32)
E_vid  = np.load(outdir/"E_vid.npy").astype(np.float32)

# reduce telemetry feature dim -> d_telem using PCA-ish linear projection (fast)
# (You can replace with nn.Linear in the model; we’ll keep it model-side to stay flexible.)
D_TELEM = telemetry_feats.shape[1]

by_script = defaultdict(list)
for idx, sid in enumerate(script_ids):
    by_script[int(sid)].append(idx)

SEQ_LEN = 5 # Changed from 32 to 5 to allow sequence creation

X_tel, X_txt, X_img, X_aud, X_vid, S_id = [], [], [], [], [], []
Y_next = []
T_txt, T_img, T_aud, T_vid = [], [], [], []

for sid, idxs in by_script.items():
    for j in range(0, len(idxs) - SEQ_LEN - 1):
        win = idxs[j:j+SEQ_LEN]
        nxt = idxs[j+SEQ_LEN]

        X_tel.append(telemetry_feats[win])   # (T, D_TELEM)
        X_txt.append(E_text[win])
        X_img.append(E_img[win])
        X_aud.append(E_aud[win])
        X_vid.append(E_vid[win])

        S_id.append(sid)
        Y_next.append(targets[nxt])

        # distill target = teacher embeddings at last timestep of window
        last = win[-1]
        T_txt.append(E_text[last]); T_img.append(E_img[last]); T_aud.append(E_aud[last]); T_vid.append(E_vid[last])

# Only stack if lists are not empty
if X_tel:
    X_tel = np.stack(X_tel).astype(np.float32)
    X_txt = np.stack(X_txt).astype(np.float32)
    X_img = np.stack(X_img).astype(np.float32)
    X_aud = np.stack(X_aud).astype(np.float32)
    X_vid = np.stack(X_vid).astype(np.float32)
    S_id  = np.array(S_id, dtype=np.int64)

    Y_next = np.stack(Y_next).astype(np.float32)
    T_txt  = np.stack(T_txt).astype(np.float32)
    T_img  = np.stack(T_img).astype(np.float32)
    T_aud  = np.stack(T_aud).astype(np.float32)
    T_vid  = np.stack(T_vid).astype(np.float32)

    print("seq dataset:", X_tel.shape, Y_next.shape, "scripts:", n_scripts)
else:
    print("No sequences could be created with the current SEQ_LEN and data.")
    X_tel, X_txt, X_img, X_aud, X_vid, S_id, Y_next, T_txt, T_img, T_aud, T_vid = (
        np.array([]) for _ in range(11)
    )

seq dataset: (28, 5, 64) (28, 3) scripts: 5


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class MMSeqDataset(Dataset):
    def __init__(self, X_tel, X_txt, X_img, X_aud, X_vid, S_id, Y_next, T_txt, T_img, T_aud, T_vid):
        self.X_tel = torch.from_numpy(X_tel)
        self.X_txt = torch.from_numpy(X_txt)
        self.X_img = torch.from_numpy(X_img)
        self.X_aud = torch.from_numpy(X_aud)
        self.X_vid = torch.from_numpy(X_vid)
        self.S_id  = torch.from_numpy(S_id)
        self.Y_next= torch.from_numpy(Y_next)
        self.T_txt = torch.from_numpy(T_txt)
        self.T_img = torch.from_numpy(T_img)
        self.T_aud = torch.from_numpy(T_aud)
        self.T_vid = torch.from_numpy(T_vid)

    def __len__(self): return self.X_tel.shape[0]
    def __getitem__(self, i):
        return (self.X_tel[i], self.X_txt[i], self.X_img[i], self.X_aud[i], self.X_vid[i],
                self.S_id[i], self.Y_next[i], self.T_txt[i], self.T_img[i], self.T_aud[i], self.T_vid[i])

def cosine_loss(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return 1.0 - (a*b).sum(dim=-1).mean()

class MultiModalStudent(nn.Module):
    def __init__(self, d_telem, n_scripts, d_model=256, nhead=8, num_layers=4,
                 d_text=384, d_img=512, d_vid=512, d_aud=768, dropout=0.1):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)

        self.p_tel  = nn.Linear(d_telem, d_model)
        self.p_text = nn.Linear(d_text, d_model)
        self.p_img  = nn.Linear(d_img, d_model)
        self.p_aud  = nn.Linear(d_aud, d_model)
        self.p_vid  = nn.Linear(d_vid, d_model)

        self.type_emb = nn.Embedding(5, d_model)  # 0=TEL 1=TXT 2=IMG 3=AUD 4=VID

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=num_layers)

        self.next_q = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 3))

        self.dist_text = nn.Linear(d_model, d_text)
        self.dist_img  = nn.Linear(d_model, d_img)
        self.dist_aud  = nn.Linear(d_model, d_aud)
        self.dist_vid  = nn.Linear(d_model, d_vid)

    def forward(self, tel_seq, txt_seq, img_seq, aud_seq, vid_seq, script_id):
        B,T,_ = tel_seq.shape
        s = self.script_emb(script_id).unsqueeze(1).unsqueeze(1)  # (B,1,1,d)

        tel = self.p_tel(tel_seq)  + self.type_emb(torch.zeros((B,T),dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        txt = self.p_text(txt_seq) + self.type_emb(torch.ones((B,T),dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        img = self.p_img(img_seq)  + self.type_emb(torch.full((B,T),2,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        aud = self.p_aud(aud_seq)  + self.type_emb(torch.full((B,T),3,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        vid = self.p_vid(vid_seq)  + self.type_emb(torch.full((B,T),4,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)

        tokens = torch.stack([tel,txt,img,aud,vid], dim=2).reshape(B, T*5, -1)
        h = self.tr(tokens)

        summary = h[:, (T-1)*5 + 0, :]  # last timestep telemetry token
        next_q = self.next_q(summary)

        dist = {
            "text": self.dist_text(summary),
            "img":  self.dist_img(summary),
            "aud":  self.dist_aud(summary),
            "vid":  self.dist_vid(summary),
        }
        return next_q, dist

torch_device = "cuda" if torch.cuda.is_available() else "cpu"

idx = np.arange(len(X_tel))
tr_idx, va_idx = train_test_split(idx, test_size=0.2, random_state=42)

ds_tr = MMSeqDataset(X_tel[tr_idx], X_txt[tr_idx], X_img[tr_idx], X_aud[tr_idx], X_vid[tr_idx],
                    S_id[tr_idx], Y_next[tr_idx], T_txt[tr_idx], T_img[tr_idx], T_aud[tr_idx], T_vid[tr_idx])
ds_va = MMSeqDataset(X_tel[va_idx], X_txt[va_idx], X_img[va_idx], X_aud[va_idx], X_vid[va_idx],
                    S_id[va_idx], Y_next[va_idx], T_txt[va_idx], T_img[va_idx], T_aud[va_idx], T_vid[va_idx])

dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
dl_va = DataLoader(ds_va, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

model = MultiModalStudent(d_telem=D_TELEM, n_scripts=n_scripts).to(torch_device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_next = nn.SmoothL1Loss()

w_txt, w_img, w_aud, w_vid = 0.2, 0.2, 0.2, 0.2

def eval_one():
    model.eval()
    tot = 0.0
    n = 0
    with torch.no_grad():
        for batch in dl_va:
            tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
            tel, txt, img, aud, vid = tel.to(torch_device), txt.to(torch_device), img.to(torch_device), aud.to(torch_device), vid.to(torch_device)
            sid, y = sid.to(torch_device), y.to(torch_device)
            tt, ti, ta, tv = tt.to(torch_device), ti.to(torch_device), ta.to(torch_device), tv.to(torch_device)

            pred_q, dist = model(tel, txt, img, aud, vid, sid)
            L = loss_next(pred_q, y)
            L = L + w_txt*cosine_loss(dist["text"], tt)
            L = L + w_img*cosine_loss(dist["img"],  ti)
            L = L + w_aud*cosine_loss(dist["aud"],  ta)
            L = L + w_vid*cosine_loss(dist["vid"],  tv)

            tot += float(L) * tel.size(0)
            n += tel.size(0)
    return tot/n

for epoch in range(1, 15000):
    model.train()
    tot = 0.0
    n = 0
    for batch in dl_tr:
        tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
        tel, txt, img, aud, vid = tel.to(torch_device), txt.to(torch_device), img.to(torch_device), aud.to(torch_device), vid.to(torch_device)
        sid, y = sid.to(torch_device), y.to(torch_device)
        tt, ti, ta, tv = tt.to(torch_device), ti.to(torch_device), ta.to(torch_device), tv.to(torch_device)

        opt.zero_grad(set_to_none=True)
        pred_q, dist = model(tel, txt, img, aud, vid, sid)

        L = loss_next(pred_q, y)
        L = L + w_txt*cosine_loss(dist["text"], tt)
        L = L + w_img*cosine_loss(dist["img"],  ti)
        L = L + w_aud*cosine_loss(dist["aud"],  ta)
        L = L + w_vid*cosine_loss(dist["vid"],  tv)

        L.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tot += float(L) * tel.size(0)
        n += tel.size(0)

    print(f"epoch {epoch:02d} train {tot/n:.5f}  val {eval_one():.5f}")

WARNING    /usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 [py.warnings]
  warnings.warn(warn_msg)



epoch 01 train 0.95606  val 1.51623
epoch 02 train 1.49244  val 1.26897
epoch 03 train 1.28061  val 0.98362
epoch 04 train 0.97899  val 0.73338
epoch 05 train 0.74418  val 0.64649
epoch 06 train 0.64078  val 0.58729
epoch 07 train 0.59950  val 0.53866
epoch 08 train 0.54868  val 0.52372
epoch 09 train 0.52715  val 0.48374
epoch 10 train 0.48002  val 0.41651
epoch 11 train 0.41950  val 0.36322
epoch 12 train 0.36897  val 0.34094
epoch 13 train 0.34962  val 0.31675
epoch 14 train 0.32875  val 0.28463
epoch 15 train 0.29264  val 0.26492
epoch 16 train 0.27169  val 0.25137
epoch 17 train 0.25983  val 0.22985
epoch 18 train 0.23990  val 0.21066
epoch 19 train 0.21793  val 0.20238
epoch 20 train 0.21020  val 0.18940
epoch 21 train 0.20062  val 0.17196
epoch 22 train 0.18584  val 0.16228
epoch 23 train 0.17032  val 0.15766
epoch 24 train 0.16767  val 0.14468
epoch 25 train 0.15545  val 0.12781
epoch 26 train 0.14024  val 0.12063
epoch 27 train 0.13465  val 0.11131
epoch 28 train 0.12813  val 

In [ ]:
from brian2 import *

def brian2_decode(pred_vec, duration_ms=200):
    v = np.array(pred_vec, dtype=float)
    v = np.clip(v, -1.0, 1.0)

    start_scope()
    defaultclock.dt = 0.1*ms
    N = 64
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''
    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')

    base = 0.6
    weights = np.linspace(0.5, 1.5, N)
    drive = base + weights*(0.2*v[0] + 0.2*v[1] + 0.2*v[2])
    G.I = drive

    M = SpikeMonitor(G)
    run(duration_ms*ms)

    rate_hz = M.num_spikes / (N * (duration_ms/1000.0))
    return float(rate_hz), M.count[:]

# demo a single sample from val set
model.eval()
batch = next(iter(dl_va))
tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
with torch.no_grad():
    pred_q, _ = model(tel[:1].to(torch_device), txt[:1].to(torch_device), img[:1].to(torch_device), aud[:1].to(torch_device), vid[:1].to(torch_device), sid[:1].to(torch_device))
pred = pred_q.cpu().numpy()[0]
rate_hz, counts = brian2_decode(pred, duration_ms=200)
pred, rate_hz

In [ ]:
# Colab cell
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install numpy pandas scikit-learn tqdm matplotlib
!pip -q install cirq-core qsimcirq
# brian2 is optional (only if you really need spiking dynamics)
!pip -q install brian2


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!mkdir -p /content/data
!unzip -q *.zip -d /content/data
!ls -lah /content/data


In [ ]:
import os, glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class HiveDataset(Dataset):
    def __init__(self, root="/content/data", split="train"):
        # Example: look for npz files like train.npz / val.npz
        npz_path = os.path.join(root, f"{split}.npz")
        if os.path.exists(npz_path):
            d = np.load(npz_path, allow_pickle=True)
            self.X = d["X"].astype(np.float32)
            self.y = d["y"]
            self.obs = d["obs"].astype(np.int64) if "obs" in d else None
            self.seq = (self.X.ndim == 3)  # (N,T,D) vs (N,D)
            return

        # Fallback: you can adapt parsing here
        raise FileNotFoundError(f"Expected {npz_path}. Export your data to train.npz/val.npz with keys X,y,(obs).")

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i])
        y = torch.tensor(self.y[i]).long() if np.issubdtype(self.y.dtype, np.integer) else torch.tensor(self.y[i]).float()
        if self.obs is None:
            obs = torch.tensor(0).long()
        else:
            obs = torch.tensor(self.obs[i]).long()
        return x, obs, y


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MLPEncoder(nn.Module):
    def __init__(self, d_in, z_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, z_dim),
            nn.LayerNorm(z_dim)
        )
    def forward(self, x):
        return self.net(x)


In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_in, h=256, z_dim=128, layers=4, heads=4, max_len=512):
        super().__init__()
        self.proj = nn.Linear(d_in, h)
        self.pos = nn.Embedding(max_len, h)
        enc_layer = nn.TransformerEncoderLayer(d_model=h, nhead=heads, batch_first=True, dropout=0.1, norm_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.to_z = nn.Sequential(nn.Linear(h, z_dim), nn.LayerNorm(z_dim))

    def forward(self, x):  # x: (B,T,D)
        B,T,D = x.shape
        t = torch.arange(T, device=x.device).unsqueeze(0).expand(B,T)
        h = self.proj(x) + self.pos(t)
        h = self.enc(h)
        pooled = h.mean(dim=1)
        return self.to_z(pooled)


In [ ]:
class FiLM(nn.Module):
    def __init__(self, z_dim, n_obs=3, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(n_obs, hidden)
        self.to_gamma = nn.Linear(hidden, z_dim)
        self.to_beta  = nn.Linear(hidden, z_dim)

    def forward(self, z, obs_id):
        h = self.emb(obs_id)
        gamma = self.to_gamma(h)
        beta = self.to_beta(h)
        return gamma * z + beta


In [ ]:
from tqdm import tqdm

def train_one_epoch(model, loader, opt, device, task="classify"):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in tqdm(loader, leave=False):
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        opt.zero_grad()

        logits, z = model(x, obs)

        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1) as y is already (batch_size, 3) and logits will be (batch_size, 3)
            loss = F.mse_loss(logits, y)
            total += y.numel()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item() * x.size(0)

    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

@torch.no_grad()
def eval_one_epoch(model, loader, device, task="classify"):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in loader:
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        logits, z = model(x, obs)
        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1)
            loss = F.mse_loss(logits, y)
            total += y.numel()
        loss_sum += loss.item() * x.size(0)
    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, loader, opt, device, task="classify"):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in tqdm(loader, leave=False):
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        opt.zero_grad()

        logits, z = model(x, obs)

        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1) as y is already (batch_size, 3) and logits will be (batch_size, 3)
            loss = F.mse_loss(logits, y)
            total += y.numel()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item() * x.size(0)

    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

@torch.no_grad()
def eval_one_epoch(model, loader, device, task="classify"):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in loader:
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        logits, z = model(x, obs)
        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1)
            loss = F.mse_loss(logits, y)
            total += y.numel()
        loss_sum += loss.item() * x.size(0)
    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class HiveModel(nn.Module):
    def __init__(self, encoder, z_dim, n_obs, n_classes, task="classify"):
        super().__init__()
        self.encoder = encoder
        self.film = FiLM(z_dim, n_obs)
        self.task = task

        if task == "classify":
            self.head = nn.Linear(z_dim, n_classes)
        elif task == "regress":
            # Corrected: Output n_classes (3) for regression
            self.head = nn.Linear(z_dim, n_classes)
        else:
            raise ValueError("Task must be 'classify' or 'regress'")

    def forward(self, x, obs_id):
        z = self.encoder(x)
        z_conditioned = self.film(z, obs_id)
        logits = self.head(z_conditioned)
        return logits, z_conditioned

In [ ]:
import os
from sklearn.model_selection import train_test_split

# Create /content/data if it doesn't exist
os.makedirs("/content/data", exist_ok=True)

# Assuming X_tel, Y_next, S_id are already defined from previous cells
# Handle the case where X_tel might be empty if no sequences were formed
if X_tel.size == 0:
    print("No data to save. X_tel is empty.")
else:
    # Split data into train and validation sets
    X_train_data, X_val_data, y_train_data, y_val_data, obs_train_data, obs_val_data = \
        train_test_split(X_tel, Y_next, S_id, test_size=0.2, random_state=42, stratify=S_id)

    # Save training data
    np.savez_compressed(
        "/content/data/train.npz",
        X=X_train_data,
        y=y_train_data,
        obs=obs_train_data
    )
    print(f"Saved /content/data/train.npz with X shape {X_train_data.shape}, y shape {y_train_data.shape}, obs shape {obs_train_data.shape}")

    # Save validation data
    np.savez_compressed(
        "/content/data/val.npz",
        X=X_val_data,
        y=y_val_data,
        obs=obs_val_data
    )
    print(f"Saved /content/data/val.npz with X shape {X_val_data.shape}, y shape {y_val_data.shape}, obs shape {obs_val_data.shape}")

# Now, proceed with model training after ensuring data is present

train_ds = HiveDataset("/content/data", "train")
val_ds   = HiveDataset("/content/data", "val")

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# detect shapes
x0, obs0, y0 = train_ds[0]
seq = (x0.ndim == 2)  # (T,D) => sequence

n_obs = train_ds.obs.max().item() + 1 if train_ds.obs is not None else 1 # dynamically get n_obs
# The target is an array of 3 floats, so it's a regression task
task = "regress" # Changed from classify

if not seq:
    d_in = x0.shape[0]
    encoder = MLPEncoder(d_in=d_in, z_dim=128)
else:
    T, d_in = x0.shape
    encoder = TransformerEncoder(d_in=d_in, h=256, z_dim=128, layers=4, heads=4, max_len=SEQ_LEN)

model = HiveModel(encoder=encoder, z_dim=128, n_obs=n_obs, n_classes=3, task=task).to(device) # n_classes=3 for quantum_host,mate,shared

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

best = 1e9
for epoch in range(1, 1000):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, device, task=task)
    va_loss, va_acc = eval_one_epoch(model, val_loader, device, task=task)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} val loss {va_loss:.4f}"
          + (f" | train acc {tr_acc:.3f} val acc {va_acc:.3f}" if task=="classify" else ""))

    if va_loss < best:
        best = va_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/hive_best.pt" if os.path.exists("/content/drive") else "/content/hive_best.pt")

In [ ]:
!pip -q install torch numpy scikit-learn tqdm


In [ ]:
import os, json, re
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit

ROOT = "/content/synth"  # Corrected path

mood_re = re.compile(r"mood=([a-zA-Z_]+)")
cls_re  = re.compile(r"classical\(H=([0-9.]+),\s*M=([0-9.]+),\s*S=([0-9.]+)\)")
q_re    = re.compile(r"quantum\(H=([0-9.]+),\s*M=([0-9.]+),\s*S=([0-9.]+)\)")

def load_manifest(path):
    rows = []
    print(f"Attempting to load manifest from: {path}")
    if not os.path.exists(path):
        print(f"Error: Path does not exist: {path}")
        # Add another check with `!ls -la` directly here to compare within Python context
        !ls -la {os.path.dirname(path)}
        raise FileNotFoundError(f"Manifest file not found at {path}")
    with open(path, "r") as f:
        for line in f:
            d = json.loads(line)
            t = d.get("text","")
            mood = mood_re.search(t).group(1)
            c = cls_re.search(t)
            q = q_re.search(t)
            classical = np.array([float(c.group(1)), float(c.group(2)), float(c.group(3))], dtype=np.float32)
            quantum   = np.array([float(q.group(1)), float(q.group(2)), float(q.group(3))], dtype=np.float32)
            rows.append({
                "i": d["i"],
                "script_name": d["script_name"],
                "cycle": d["cycle"],
                "mood": mood,
                "classical": classical,
                "quantum": quantum
            })
    return rows

rows = load_manifest(os.path.join(ROOT, "manifest.jsonl"))
print("rows:", len(rows), "unique scripts:", sorted(set(r["script_name"] for r in rows)), "unique moods:", sorted(set(r["mood"] for r in rows)))

E_text = np.load(os.path.join(ROOT, "E_text.npy")).astype(np.float32)
E_img  = np.load(os.path.join(ROOT, "E_img.npy")).astype(np.float32)
E_vid  = np.load(os.path.join(ROOT, "E_vid.npy")).astype(np.float32)
E_aud  = np.load(os.path.join(ROOT, "E_aud.npy")).astype(np.float32)

assert E_text.shape[0] == len(rows) == E_img.shape[0] == E_vid.shape[0] == E_aud.shape[0]

# label maps
scripts = sorted(set(r["script_name"] for r in rows))
script2id = {s:i for i,s in enumerate(scripts)}

moods = sorted(set(r["mood"] for r in rows))
mood2id = {m:i for i,m in enumerate(moods)}

y_mood = np.array([mood2id[r["mood"]] for r in rows], dtype=np.int64)
obs_id = np.array([script2id[r["script_name"]] for r in rows], dtype=np.int64)

Y_classical = np.stack([r["classical"] for r in rows], axis=0)
Y_quantum   = np.stack([r["quantum"] for r in rows], axis=0)

# stratified split by mood (since tiny dataset)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(y_mood)), y_mood))
print("train/val:", len(train_idx), len(val_idx))

# Define n_obs and n_moods globally
n_obs = len(scripts)
n_moods = len(moods)

In [ ]:
import os

file_path = "/content/data/synth/manifest.jsonl"
if os.path.exists(file_path):
    print(f"The file {file_path} exists.")
else:
    print(f"The file {file_path} does NOT exist.")

In [ ]:
class SynthHive(Dataset):
    def __init__(self, idx):
        self.idx = idx

    def __len__(self): return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        x = {
            "text": torch.from_numpy(E_text[i]),
            "img":  torch.from_numpy(E_img[i]),
            "vid":  torch.from_numpy(E_vid[i]),
            "aud":  torch.from_numpy(E_aud[i]),
        }
        obs = torch.tensor(obs_id[i]).long()
        mood = torch.tensor(y_mood[i]).long()
        classical = torch.from_numpy(Y_classical[i])
        quantum   = torch.from_numpy(Y_quantum[i])
        return x, obs, mood, classical, quantum

train_ds = SynthHive(train_idx)
val_ds   = SynthHive(val_idx)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ModEncoder(nn.Module):
    def __init__(self, d_in, z=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, z),
            nn.LayerNorm(z)
        )
    def forward(self, x): return self.net(x)

class FiLM(nn.Module):
    def __init__(self, z=128, n_obs=5, h=128):
        super().__init__()
        self.emb = nn.Embedding(n_obs, h)
        self.gamma = nn.Linear(h, z)
        self.beta  = nn.Linear(h, z)
    def forward(self, zvec, obs):
        e = self.emb(obs)
        return self.gamma(e) * zvec + self.beta(e)

class HiveFusion(nn.Module):
    """
    variant:
      - single modality: pass mods=["text"] etc
      - fused: mods=["text","img","vid","aud"]
    """
    def __init__(self, mods, n_obs, n_moods, z=128, multitask=True):
        super().__init__()
        self.mods = mods
        self.z = z
        self.multitask = multitask

        dims = {"text":384, "img":512, "vid":512, "aud":768}
        self.enc = nn.ModuleDict({m: ModEncoder(dims[m], z=z) for m in mods})

        # fuse by concat then project back to z
        self.fuse = nn.Sequential(
            nn.Linear(len(mods)*z, z),
            nn.GELU(),
            nn.LayerNorm(z)
        )

        self.film = FiLM(z=z, n_obs=n_obs)

        # heads
        self.mood_head = nn.Linear(z, n_moods)
        if multitask:
            self.classical_head = nn.Linear(z, 3)
            self.quantum_head   = nn.Linear(z, 3)

    def forward(self, xdict, obs):
        zs = [self.enc[m](xdict[m]) for m in self.mods]
        zcat = torch.cat(zs, dim=-1)
        z = self.fuse(zcat)
        z = self.film(z, obs)
        mood_logits = self.mood_head(z)

        if self.multitask:
            return mood_logits, self.classical_head(z), self.quantum_head(z), z
        return mood_logits, None, None, z

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct = 0, 0
    loss_sum = 0.0
    for x, obs, mood, classical, quantum in loader:
        x = {k:v.to(device) for k,v in x.items()}
        obs = obs.to(device); mood = mood.to(device)
        classical = classical.to(device); quantum = quantum.to(device)

        logits, c_hat, q_hat, _ = model(x, obs) # Unpack all 4 values, ignoring the latent 'z'
        loss = F.cross_entropy(logits, mood)
        if c_hat is not None:
            loss = loss + 0.25*F.mse_loss(c_hat, classical) + 0.25*F.mse_loss(q_hat, quantum)

        pred = logits.argmax(-1)
        correct += (pred == mood).sum().item()
        total += mood.numel()
        loss_sum += loss.item() * mood.size(0)

    return loss_sum/len(loader.dataset), correct/total

def train(model, train_loader, val_loader, device, epochs=40, lr=3e-4, save_path="hive_best.pt"):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    best = 1e9
    for ep in range(1, epochs+1):
        model.train()
        for x, obs, mood, classical, quantum in train_loader:
            x = {k:v.to(device) for k,v in x.items()}
            obs = obs.to(device); mood = mood.to(device)
            classical = classical.to(device); quantum = quantum.to(device)

            opt.zero_grad()
            logits, c_hat, q_hat, _ = model(x, obs) # Unpack all 4 values, ignoring the latent 'z'
            loss = F.cross_entropy(logits, mood)
            if c_hat is not None:
                loss = loss + 0.25*F.mse_loss(c_hat, classical) + 0.25*F.mse_loss(q_hat, quantum)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        vloss, vacc = evaluate(model, val_loader, device)
        if vloss < best:
            best = vloss
            torch.save(model.state_dict(), save_path)

        if ep % 5 == 0 or ep == 1:
            print(f"ep {ep:02d} | val loss {vloss:.4f} | val acc {vacc:.3f}")

    return best

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def run_all():
    outdir = "/content/drive/MyDrive" if os.path.exists("/content/drive") else "/content"
    n_obs = len(scripts)
    n_moods = len(moods)

    variants = [
        (["text"], "text_only"),
        (["img"],  "img_only"),
        (["vid"],  "vid_only"),
        (["aud"],  "aud_only"),
        (["text","img","vid","aud"], "fused_all"),
    ]

    results = {}
    for mods, name in variants:
        print("\n===", name, mods, "===")
        model = HiveFusion(mods=mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)
        save_path = os.path.join(outdir, f"hive_{name}.pt")
        best = train(model, train_loader, val_loader, device, epochs=40, lr=3e-4, save_path=save_path)
        vloss, vacc = evaluate(model, val_loader, device)
        results[name] = {"best_val_loss": float(best), "final_val_acc": float(vacc), "ckpt": save_path}
        print("saved:", save_path, "| final val acc:", vacc)

    return results

results = run_all()
results

In [ ]:
!ls -la /content/synth

In [ ]:
import os, random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
from sklearn.model_selection import StratifiedKFold

N = len(y_mood)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(np.zeros(N), y_mood))
print("kfold folds:", len(folds), "N:", N)


In [ ]:
from torch.utils.data import WeightedRandomSampler

def make_balanced_sampler(idx, labels):
    y = labels[idx]
    classes, counts = np.unique(y, return_counts=True)
    count_map = {c:cnt for c,cnt in zip(classes, counts)}
    w = np.array([1.0 / count_map[yy] for yy in y], dtype=np.float64)
    return WeightedRandomSampler(weights=w, num_samples=len(w), replacement=True)


In [ ]:
import torch.nn.functional as F

def info_nce(z, mood, temperature=0.2):
    """
    z: (B,Z) latent
    mood: (B,) int
    """
    z = F.normalize(z, dim=-1)
    sim = (z @ z.T) / temperature  # (B,B)

    # mask self
    B = z.size(0)
    self_mask = torch.eye(B, device=z.device).bool()
    sim = sim.masked_fill(self_mask, -1e9)

    # positives: same mood
    pos = (mood.unsqueeze(1) == mood.unsqueeze(0)) & (~self_mask)
    if pos.sum() == 0:
        return z.new_tensor(0.0)

    # log-softmax over rows
    logp = F.log_softmax(sim, dim=1)

    # average log prob of positives for each anchor that has positives
    pos_counts = pos.sum(dim=1)
    valid = pos_counts > 0
    loss = -(logp[pos].view(B, -1).sum(dim=1) / pos_counts.clamp_min(1)).masked_select(valid).mean()
    return loss


In [ ]:
import torch.nn.functional as F

def info_nce(z, mood, temperature=0.2):
    """
    z: (B,Z) latent
    mood: (B,) int
    """
    z = F.normalize(z, dim=-1)
    sim = (z @ z.T) / temperature  # (B,B)

    # mask self
    B = z.size(0)
    self_mask = torch.eye(B, device=z.device).bool()
    sim = sim.masked_fill(self_mask, -1e9)

    # positives: same mood
    pos = (mood.unsqueeze(1) == mood.unsqueeze(0)) & (~self_mask)
    if pos.sum() == 0:
        return z.new_tensor(0.0)

    # log-softmax over rows
    logp = F.log_softmax(sim, dim=1)

    # Calculate sum of log probabilities for positive pairs for each anchor
    sum_logp_pos = (logp * pos.float()).sum(dim=1) # (B,)

    # Divide by pos_counts (number of positives per anchor)
    pos_counts = pos.sum(dim=1) # (B,)

    # Take mean only for valid anchors (those with at least one positive)
    valid = pos_counts > 0
    per_anchor_loss = - (sum_logp_pos / pos_counts.clamp_min(1)) # (B,)

    loss = per_anchor_loss.masked_select(valid).mean()
    return loss


In [ ]:
# Replace forward in HiveFusion with this version
def forward(self, xdict, obs):
    zs = [self.enc[m](xdict[m]) for m in self.mods]
    zcat = torch.cat(zs, dim=-1)
    z = self.fuse(zcat)
    z = self.film(z, obs)
    mood_logits = self.mood_head(z)

    if self.multitask:
        return mood_logits, self.classical_head(z), self.quantum_head(z), z
    return mood_logits, None, None, z


In [ ]:
import torch.nn as nn
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader, device, loss_weights):
    model.eval()
    total, correct = 0, 0
    loss_sum = 0.0

    w_cls, w_c, w_q, w_con = loss_weights

    for x, obs, mood, classical, quantum in loader:
        x = {k:v.to(device) for k,v in x.items()}
        obs = obs.to(device); mood = mood.to(device)
        classical = classical.to(device); quantum = quantum.to(device)

        logits, c_hat, q_hat, z = model(x, obs)

        loss = w_cls * F.cross_entropy(logits, mood)
        if c_hat is not None:
            loss = loss + w_c*F.mse_loss(c_hat, classical) + w_q*F.mse_loss(q_hat, quantum)
        if w_con > 0:
            loss = loss + w_con*info_nce(z, mood)

        pred = logits.argmax(-1)
        correct += (pred == mood).sum().item()
        total += mood.numel()
        loss_sum += loss.item() * mood.size(0)

    return loss_sum/len(loader.dataset), correct/total


def train_fold(model, train_loader, val_loader, device,
               epochs=120, lr=3e-4, save_path="hive_best.pt",
               loss_weights=(1.0, 0.25, 0.25, 0.10),
               patience=20):
    """
    loss_weights = (w_cls, w_classical_mse, w_quantum_mse, w_contrastive)
    """
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    best = 1e9
    bad = 0

    for ep in range(1, epochs+1):
        model.train()
        for x, obs, mood, classical, quantum in train_loader:
            x = {k:v.to(device) for k,v in x.items()}
            obs = obs.to(device); mood = mood.to(device)
            classical = classical.to(device); quantum = quantum.to(device)

            opt.zero_grad()
            logits, c_hat, q_hat, z = model(x, obs)

            w_cls, w_c, w_q, w_con = loss_weights
            loss = w_cls * F.cross_entropy(logits, mood)
            loss = loss + w_c*F.mse_loss(c_hat, classical) + w_q*F.mse_loss(q_hat, quantum)
            if w_con > 0:
                loss = loss + w_con*info_nce(z, mood)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        vloss, vacc = evaluate(model, val_loader, device, loss_weights)

        if vloss < best - 1e-5:
            best = vloss
            bad = 0
            torch.save(model.state_dict(), save_path)
        else:
            bad += 1

        if ep % 10 == 0 or ep == 1:
            print(f"ep {ep:03d} | val loss {vloss:.4f} | val acc {vacc:.3f} | bad {bad}/{patience}")

        if bad >= patience:
            break

    return best


In [ ]:
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def run_all_kfold():
    outdir = "/content/drive/MyDrive" if os.path.exists("/content/drive") else "/content"
    n_obs = len(scripts)
    n_moods = len(moods)

    variants = [
        (["text"], "text_only"),
        (["img"],  "img_only"),
        (["vid"],  "vid_only"),
        (["aud"],  "aud_only"),
        (["text","img","vid","aud"], "fused_all"),
    ]

    # weights: (CE mood, MSE classical, MSE quantum, contrastive)
    loss_weights = (1.0, 0.25, 0.25, 0.10)

    summary = {}
    for mods, name in variants:
        fold_metrics = []
        print("\n==============================")
        print("VARIANT:", name, mods)

        for fi, (tr_idx, va_idx) in enumerate(folds, start=1):
            # loaders
            train_ds = SynthHive(tr_idx)
            val_ds   = SynthHive(va_idx)

            # optional balancing:
            sampler = make_balanced_sampler(tr_idx, y_mood)
            train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)
            val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

            # model
            model = HiveFusion(mods=mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)

            save_path = os.path.join(outdir, f"hive_{name}_fold{fi}.pt")
            best = train_fold(
                model, train_loader, val_loader, device,
                epochs=200, lr=3e-4,
                save_path=save_path,
                loss_weights=loss_weights,
                patience=25
            )

            vloss, vacc = evaluate(model, val_loader, device, loss_weights)
            fold_metrics.append((float(vloss), float(vacc), save_path))
            print(f"fold {fi}: val_loss={vloss:.4f} val_acc={vacc:.3f} saved={save_path}")

        # aggregate
        losses = [m[0] for m in fold_metrics]
        accs   = [m[1] for m in fold_metrics]
        summary[name] = {
            "mods": mods,
            "mean_val_loss": float(np.mean(losses)),
            "std_val_loss":  float(np.std(losses)),
            "mean_val_acc":  float(np.mean(accs)),
            "std_val_acc":   float(np.std(accs)),
            "checkpoints":   [m[2] for m in fold_metrics],
        }

        print(f"\n{name} | mean acc {summary[name]['mean_val_acc']:.3f} \u00b1 {summary[name]['std_val_acc']:.3f}")

    return summary

summary = run_all_kfold()
summary

In [ ]:
import numpy as np

# --- Quantum: Cirq (works even if certain helper APIs are missing) ---
import cirq

def z_expectations_from_state(psi, n_qubits):
    """Compute <Z_i> for each qubit i from a full statevector psi."""
    # psi: complex statevector length 2^n
    probs = np.abs(psi)**2
    exps = []
    for i in range(n_qubits):
        # Z expectation: sum_{bitstring} (+1 if bit i=0 else -1) * P(bitstring)
        # bit i corresponds to position (n_qubits-1-i) in binary indexing depending on convention;
        # Cirq uses little-endian ordering in many contexts. We'll match that:
        exp = 0.0
        for idx, p in enumerate(probs):
            bit = (idx >> i) & 1  # little-endian
            exp += (1.0 if bit == 0 else -1.0) * p
        exps.append(exp)
    return np.array(exps, dtype=np.float32)

def quantum_encode(features, n_qubits=4, depth=2):
    """
    Map features -> parameterized circuit -> expectation vector in [-1,1]^n_qubits
    """
    qubits = cirq.LineQubit.range(n_qubits)
    circuit = cirq.Circuit()

    # simple feature embedding: Ry rotations on each qubit
    # features assumed length >= n_qubits
    for i, q in enumerate(qubits):
        theta = float(features[i]) * np.pi
        circuit.append(cirq.ry(theta)(q))

    # entangling layers
    for _ in range(depth):
        for i in range(n_qubits - 1):
            circuit.append(cirq.CNOT(qubits[i], qubits[i+1]))
        for i, q in enumerate(qubits):
            circuit.append(cirq.rz(0.25 * np.pi)(q))

    sim = cirq.Simulator()
    result = sim.simulate(circuit)
    psi = np.array(result.final_state_vector, dtype=np.complex64)
    return z_expectations_from_state(psi, n_qubits)

# --- Hive bridge: tiny MLP (replace with your multimodal+observer model later) ---
import torch
import torch.nn as nn

class HiveBridge(nn.Module):
    def __init__(self, d_in, d_q, z_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + d_q, 128),
            nn.GELU(),
            nn.Linear(128, z_dim),
            nn.LayerNorm(z_dim),
        )
        # translate to Brian2 drive (current) and synapse delta
        self.to_current = nn.Linear(z_dim, 1)      # scalar neuromod drive
        self.to_synapse = nn.Linear(z_dim, 1)      # global synapse gain (demo)

    def forward(self, x, q):
        h = torch.cat([x, q], dim=-1)
        z = self.net(h)
        drive = self.to_current(z)   # (B,1)
        sgain = self.to_synapse(z)   # (B,1)
        return z, drive, sgain

# --- Brian2 substrate ---
from brian2 import *

def run_brian2_hive(drive_scalar, syn_gain, duration_ms=500):
    """
    drive_scalar: float (from Hive) ~ arbitrary
    syn_gain: float (from Hive) ~ arbitrary
    returns: spike times, a simple haptic envelope
    """
    start_scope()

    N = 50
    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')
    G.v = 0

    # baseline + Hive drive
    base = 0.6
    drive = float(drive_scalar)
    G.I = base + 0.4*np.tanh(drive)  # keep stable

    # recurrent synapses with Hive gain
    # Declaring 'w' in the Synapses model
    S = Synapses(G, G, model='w : 1', on_pre='v_post += w')
    S.connect(p=0.1)
    S.w = (0.02 + 0.03*np.tanh(float(syn_gain)))  # small excitatory kick

    spikemon = SpikeMonitor(G)
    ratemon = PopulationRateMonitor(G)

    run(duration_ms*ms)

    # "haptic envelope": use population rate (Hz) scaled 0..1
    rate = ratemon.smooth_rate(window='flat', width=20*ms) / Hz
    env = np.clip(rate / 100.0, 0, 1)  # pretend 100 Hz = max vibration
    return spikemon.t/ms, spikemon.i, env

# --- Demo pipeline ---
# input features (your “cognition” vector) – replace with embeddings/sentiment features
x_feat = np.random.randn(8).astype(np.float32)

# quantum vector from first 4 features
q_vec = quantum_encode(x_feat, n_qubits=4, depth=2)

# Hive forward
device = "cuda" if torch.cuda.is_available() else "cpu"
hive = HiveBridge(d_in=8, d_q=4, z_dim=64).to(device).eval()

with torch.no_grad():
    x_t = torch.tensor(x_feat[None, :], device=device)
    q_t = torch.tensor(q_vec[None, :], device=device)
    z, drive, sgain = hive(x_t, q_t)

drive_scalar = float(drive.cpu().numpy().squeeze())
syn_gain = float(sgain.cpu().numpy().squeeze())

# Brian2 run
t_spk, i_spk, haptic_env = run_brian2_hive(drive_scalar, syn_gain, duration_ms=500)

print("Quantum q:", q_vec)
print("Hive drive:", drive_scalar, "syn_gain:", syn_gain)
print("Spikes:", len(t_spk), "Haptic env samples:", len(haptic_env))


In [ ]:
!pip -q install numpy torch brian2 cirq-core qsimcirq


In [ ]:
import torch, torch.nn as nn

class HiveBridge(nn.Module):
    """
    Input: x (your cognition/features), q (quantum expectations)
    Output:
      - z: latent cognition state
      - drive_vec: per-population drive currents (scaled later)
      - syn_gain: global synapse gain (for option 2)
      - gate: neuromodulator/plasticity gate (for option 3)
    """
    def __init__(self, d_in, d_q, z_dim=64, n_pops=3):
        super().__init__()
        self.core = nn.Sequential(
            nn.Linear(d_in + d_q, 128),
            nn.GELU(),
            nn.Linear(128, z_dim),
            nn.LayerNorm(z_dim),
        )
        self.drive = nn.Linear(z_dim, n_pops)  # per-module drive
        self.syn_gain = nn.Linear(z_dim, 1)    # global gain
        self.gate = nn.Linear(z_dim, 1)        # plasticity gate

    def forward(self, x, q):
        z = self.core(torch.cat([x, q], dim=-1))
        drive_vec = self.drive(z)
        syn_gain = self.syn_gain(z)
        gate = self.gate(z)
        return z, drive_vec, syn_gain, gate


In [ ]:
from brian2 import *

def run_brian2_drive_only(drive_vec, duration_ms=800, N=60):
    """
    drive_vec: np array shape (3,) from HiveBridge
    Returns: spikes + "haptic envelope" from motor population rate
    """
    start_scope()
    defaultclock.dt = 1*ms

    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    # 3 modules: sensory, affective, motor
    Gs = [NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler') for _ in range(3)]
    for G in Gs:
        G.v = 0

    # Map Hive drive -> currents (clipped for stability)
    base = [0.55, 0.55, 0.55]
    drive = np.tanh(np.array(drive_vec, dtype=np.float32))  # [-1,1]
    scales = [0.25, 0.30, 0.35]  # motor slightly more sensitive

    for k, G in enumerate(Gs):
        G.I = base[k] + scales[k] * float(drive[k])

    # Mild recurrent within each pop (fixed)
    for G in Gs:
        S = Synapses(G, G, model='w : 1', on_pre='v_post += w')
        S.connect(p=0.08)
        S.w = 0.02

    spk = [SpikeMonitor(G) for G in Gs]
    rate_motor = PopulationRateMonitor(Gs[2])

    run(duration_ms*ms)

    # Haptic envelope: motor population smoothed rate, normalized
    rate = rate_motor.smooth_rate(window='flat', width=25*ms) / Hz
    env = np.clip(rate / 120.0, 0, 1)  # 120 Hz -> max "vibe"
    return spk, env

In [ ]:
def run_brian2_drive_plus_syn_gain(drive_vec, syn_gain, duration_ms=800, N=60):
    start_scope()
    defaultclock.dt = 1*ms

    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    Gs = [NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler') for _ in range(3)]
    for G in Gs:
        G.v = 0

    base = [0.55, 0.55, 0.55]
    drive = np.tanh(np.array(drive_vec, dtype=np.float32))
    scales = [0.25, 0.30, 0.35]
    for k, G in enumerate(Gs):
        G.I = base[k] + scales[k] * float(drive[k])

    # synapse gain in a safe range
    g = float(np.tanh(float(syn_gain)))  # [-1,1]
    gain = 0.02 + 0.03 * (g + 1)/2.0     # ~ [0.02, 0.05]

    # Intra-module
    for G in Gs:
        S = Synapses(G, G, model='w : 1', on_pre='v_post += w')
        S.connect(p=0.08)
        S.w = 0.02

    # Inter-module: sensory->affective->motor (translated by Hive syn_gain)
    Sa = Synapses(Gs[0], Gs[1], model='w : 1', on_pre='v_post += w'); Sa.connect(p=0.10); Sa.w = gain
    Am = Synapses(Gs[1], Gs[2], model='w : 1', on_pre='v_post += w'); Am.connect(p=0.12); Am.w = gain

    spk = [SpikeMonitor(G) for G in Gs]
    rate_motor = PopulationRateMonitor(Gs[2])

    run(duration_ms*ms)

    rate = rate_motor.smooth_rate(window='flat', width=25*ms) / Hz
    env = np.clip(rate / 120.0, 0, 1)
    return spk, env, gain

In [ ]:
def run_brian2_drive_plus_gated_stdp(drive_vec, gate, duration_ms=2000, N=60):
    start_scope()
    defaultclock.dt = 1*ms

    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    # 3 modules
    Gs = [NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler') for _ in range(3)]
    for G in Gs:
        G.v = 0

    base = [0.55, 0.55, 0.55]
    drive = np.tanh(np.array(drive_vec, dtype=np.float32))
    scales = [0.25, 0.30, 0.35]
    for k, G in enumerate(Gs):
        G.I = base[k] + scales[k] * float(drive[k])

    # neuromodulator gate in [0,1]
    g = float(1 / (1 + np.exp(-float(gate))))  # sigmoid

    # STDP parameters
    tau_pre = 20*ms
    tau_post = 20*ms
    A_pre = 0.01 * g
    A_post = -0.012 * g
    wmin, wmax = 0.0, 0.08

    # Plastic synapses: affective -> motor
    S = Synapses(
        Gs[1], Gs[2],
        model="""
        w : 1
        dpre/dt = -pre/tau_pre : 1 (event-driven)
        dpost/dt = -post/tau_post : 1 (event-driven)
        """,
        on_pre="""
        v_post += w
        pre += A_pre
        w = clip(w + post, wmin, wmax)
        """,
        on_post="""
        post += A_post
        w = clip(w + pre, wmin, wmax)
        """,
        method='euler'
    )
    S.connect(p=0.12)
    S.w = 0.02

    # Fixed sensory->affective so sensory can stimulate learning downstream
    Sa = Synapses(Gs[0], Gs[1], model='w : 1', on_pre='v_post += w')
    Sa.connect(p=0.10)
    Sa.w = 0.03

    spk = [SpikeMonitor(G) for G in Gs]
    rate_motor = PopulationRateMonitor(Gs[2])

    run(duration_ms*ms)

    rate = rate_motor.smooth_rate(window='flat', width=30*ms) / Hz
    env = np.clip(rate / 120.0, 0, 1)
    return spk, env, g, float(np.mean(S.w))

In [ ]:
import numpy as np
import torch

def run_pipeline(x_feat, n_qubits=4):
    # quantum
    q = quantum_encode(x_feat, n_qubits=n_qubits, depth=2)

    # hive
    device = "cuda" if torch.cuda.is_available() else "cpu"
    hive = HiveBridge(d_in=len(x_feat), d_q=n_qubits, z_dim=64, n_pops=3).to(device).eval()

    with torch.no_grad():
        x_t = torch.tensor(x_feat[None, :], device=device, dtype=torch.float32)
        q_t = torch.tensor(q[None, :], device=device, dtype=torch.float32)
        z, drive_vec, syn_gain, gate = hive(x_t, q_t)

    drive_vec = drive_vec.cpu().numpy().squeeze()
    syn_gain = float(syn_gain.cpu().numpy().squeeze())
    gate = float(gate.cpu().numpy().squeeze())

    # Brian2 option 1
    spk1, env1 = run_brian2_drive_only(drive_vec)

    # Brian2 option 2
    spk2, env2, gain_used = run_brian2_drive_plus_syn_gain(drive_vec, syn_gain)

    # Brian2 option 3
    spk3, env3, gate_used, mean_w = run_brian2_drive_plus_gated_stdp(drive_vec, gate)

    return {
        "q": q,
        "drive_vec": drive_vec,
        "syn_gain_raw": syn_gain,
        "syn_gain_used": gain_used,
        "gate_raw": gate,
        "gate_used": gate_used,
        "mean_plastic_w": mean_w,
        "env1": env1,
        "env2": env2,
        "env3": env3,
        "spk1": spk1,
        "spk2": spk2,
        "spk3": spk3,
    }

# demo input (replace with your fused embeddings / sentiment vector slice)
x_feat = np.random.randn(8).astype(np.float32)
out = run_pipeline(x_feat)
print("q:", out["q"])
print("drive_vec:", out["drive_vec"])
print("syn_gain_used:", out["syn_gain_used"], "gate_used:", out["gate_used"], "mean_plastic_w:", out["mean_plastic_w"])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

class ZProject(nn.Module):
    """Project student z -> teacher z dim so we can match features."""
    def __init__(self, z_s, z_t):
        super().__init__()
        self.proj = nn.Linear(z_s, z_t)
    def forward(self, z_s):
        return self.proj(z_s)

def distill_train(
    teacher, student, train_loader, val_loader, device,
    epochs=20, lr=3e-4,
    T=2.0,                 # temperature for soft labels
    w_hard=1.0,            # weight for hard label loss
    w_soft=0.5,            # weight for soft KL
    w_reg=0.25,            # weight for classical/quantum regressions (if enabled)
    w_z=0.2,               # weight for feature distill
    save_path="student_distilled.pt"
):
    teacher = teacher.to(device).eval()
    student = student.to(device).train()

    # infer z dims once (quickly) by a single batch
    x0, obs0, mood0, classical0, quantum0 = next(iter(train_loader))
    x0 = {k:v.to(device) for k,v in x0.items()}
    obs0 = obs0.to(device)
    with torch.no_grad():
        _, _, _, zt = teacher(x0, obs0)
        _, _, _, zs = student(x0, obs0)
    zproj = ZProject(z_s=zs.shape[-1], z_t=zt.shape[-1]).to(device)

    opt = torch.optim.AdamW(list(student.parameters()) + list(zproj.parameters()),
                            lr=lr, weight_decay=1e-2)

    best = 1e9
    for ep in range(1, epochs + 1):
        student.train()
        zproj.train()
        loss_sum = 0.0

        for x, obs, mood, classical, quantum in tqdm(train_loader, desc=f"distill ep {ep}", leave=False):
            x = {k:v.to(device) for k,v in x.items()}
            obs = obs.to(device)
            mood = mood.to(device)
            classical = classical.to(device)
            quantum = quantum.to(device)

            with torch.no_grad():
                t_logits, t_c, t_q, t_z = teacher(x, obs)

            s_logits, s_c, s_q, s_z = student(x, obs)

            # 1) hard supervised loss
            hard = F.cross_entropy(s_logits, mood)

            # 2) soft distillation loss (KL between teacher/student mood distributions)
            # KL(student || teacher) or KL(teacher || student) both ok; common is student vs teacher
            t_probs = F.softmax(t_logits / T, dim=-1)
            s_logp  = F.log_softmax(s_logits / T, dim=-1)
            soft = F.kl_div(s_logp, t_probs, reduction="batchmean") * (T * T)

            # 3) regression (if your model returns these)
            reg = 0.0
            if (s_c is not None) and (s_q is not None) and (t_c is not None) and (t_q is not None):
                # keep student aligned with ground-truth, optionally also match teacher
                reg = F.mse_loss(s_c, classical) + F.mse_loss(s_q, quantum)

            # 4) feature distillation: match latent z
            z_loss = F.mse_loss(zproj(s_z), t_z)

            loss = w_hard*hard + w_soft*soft + w_reg*reg + w_z*z_loss

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()

            loss_sum += loss.item() * mood.size(0)

        train_loss = loss_sum / len(train_loader.dataset)

        # quick val (hard-only) to choose best student
        student.eval()
        zproj.eval()
        vloss, vacc = 0.0, 0.0
        total, correct = 0, 0
        with torch.no_grad():
            for x, obs, mood, classical, quantum in val_loader:
                x = {k:v.to(device) for k,v in x.items()}
                obs = obs.to(device)
                mood = mood.to(device)
                classical = classical.to(device)
                quantum = quantum.to(device)
                logits, c_hat, q_hat, _ = student(x, obs)
                loss = F.cross_entropy(logits, mood)
                if c_hat is not None and q_hat is not None:
                    loss = loss + 0.25*F.mse_loss(c_hat, classical) + 0.25*F.mse_loss(q_hat, quantum)
                pred = logits.argmax(-1)
                correct += (pred == mood).sum().item()
                total += mood.numel()
                vloss += loss.item() * mood.size(0)

        vloss = vloss / len(val_loader.dataset)
        vacc = correct / max(1, total)

        print(f"ep {ep:02d} train={train_loss:.4f} val={vloss:.4f} val_acc={vacc:.3f}")

        if vloss < best:
            best = vloss
            torch.save({"student": student.state_dict(), "zproj": zproj.state_dict()}, save_path)
            print("  saved:", save_path)

    return save_path


In [ ]:
!nvidia-smi
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device


In [ ]:
!pip -q install --upgrade pip
!pip -q install "transformers>=4.38" accelerate safetensors einops
!pip -q install sentence-transformers
!pip -q install open-clip-torch pillow
!pip -q install librosa soundfile


In [ ]:
!pip -q install decord av


In [ ]:
TEXT_MODEL  = "intfloat/e5-base-v2"               # text embeddings
AUDIO_MODEL = "facebook/wav2vec2-base-960h"       # audio embeddings
# OpenCLIP uses public checkpoints; no key needed:
IMAGE_MODEL = ("ViT-B-32", "laion2b_s34b_b79k")   # image embeddings


In [ ]:
import numpy as np
import torch

# ---- Text ----
from sentence_transformers import SentenceTransformer
text_encoder = SentenceTransformer(TEXT_MODEL, device=device)

# ---- Image (OpenCLIP) ----
import open_clip
from PIL import Image
image_encoder, _, image_preprocess = open_clip.create_model_and_transforms(
    IMAGE_MODEL[0], pretrained=IMAGE_MODEL[1], device=device
)
image_encoder.eval()

# ---- Audio ----
from transformers import AutoProcessor, AutoModel
audio_processor = AutoProcessor.from_pretrained(AUDIO_MODEL)
audio_encoder = AutoModel.from_pretrained(AUDIO_MODEL).to(device).eval()

print("Loaded:", TEXT_MODEL, IMAGE_MODEL, AUDIO_MODEL)


In [ ]:
import librosa

def embed_text(texts, batch_size=32, normalize=True):
    # For E5 models, prefixing helps: "query: ..." / "passage: ..."
    # If your texts are “content”, use passage:
    texts2 = [("passage: " + t) for t in texts]
    emb = text_encoder.encode(
        texts2, batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=normalize,
        show_progress_bar=True
    )
    return emb.astype(np.float32)

@torch.no_grad()
def embed_images(image_paths, batch_size=32, normalize=True):
    vecs = []
    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i+batch_size]
        imgs = [image_preprocess(Image.open(p).convert("RGB")) for p in batch]
        x = torch.stack(imgs).to(device)
        z = image_encoder.encode_image(x)
        if normalize:
            z = z / z.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        vecs.append(z.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(vecs, axis=0)

@torch.no_grad()
def embed_audio(audio_paths, target_sr=16000, batch_size=8, normalize=True):
    vecs = []
    for i in range(0, len(audio_paths), batch_size):
        batch = audio_paths[i:i+batch_size]
        waves = []
        for p in batch:
            w, _ = librosa.load(p, sr=target_sr, mono=True)
            waves.append(w)

        inputs = audio_processor(waves, sampling_rate=target_sr, return_tensors="pt", padding=True)
        inputs = {k:v.to(device) for k,v in inputs.items()}
        out = audio_encoder(**inputs)
        z = out.last_hidden_state.mean(dim=1)  # (B,H)
        if normalize:
            z = z / z.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        vecs.append(z.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(vecs, axis=0)


In [ ]:
import pandas as pd
from pathlib import Path

# The synthetic data is generated directly in /content/synth
# and the manifest.jsonl and .npy embeddings are already created and loaded.
# The `records` variable from cell bydqcFAki_ev contains the parsed manifest data.
# The `E_text`, `E_img`, `E_aud`, `E_vid` arrays are already loaded.

# Create a DataFrame from the `records` list (which contains parsed manifest.jsonl data)
df = pd.DataFrame(records)

# Adjust column names based on the keys available in `records`
# The actual text content is in the 'text' column of the dataframe
TEXT_COL  = "text"
IMG_COL   = "image_path"
AUD_COL   = "audio_path"
MOOD_COL  = "mood"  # The 'mood' extracted from text is directly in this column

# 'classical' and 'quantum' are stored as numpy arrays within the DataFrame cells
CLASS_COL = "classical"
QUANT_COL = "quantum"

# 'script_id' from records corresponds to the observer ID
OBS_COLS = [c for c in df.columns if c.startswith("script_id")] # 'script_id' is directly available

# The original text and paths are now accessible directly from df
texts    = df[TEXT_COL].tolist()
img_paths = df[IMG_COL].tolist()
aud_paths = df[AUD_COL].tolist()

print("samples:", len(df), "obs_cols:", OBS_COLS)
print("First 5 rows of the DataFrame:")
print(df.head())

In [ ]:
# Define the teacher model (fused_all variant from previous k-fold training)
teacher_mods = ["text", "img", "vid", "aud"]
teacher_model = HiveFusion(mods=teacher_mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)
teacher_ckpt_path = summary["fused_all"]["checkpoints"][0] # Using the checkpoint from the first fold
teacher_model.load_state_dict(torch.load(teacher_ckpt_path, map_location=device))
teacher_model = teacher_model.to(device).eval()

# Define the student model (e.g., text_only or aud_only, as they performed less well but are simpler)
student_mods = ["aud"] # Let's try to distill knowledge into the audio-only model
student_model = HiveFusion(mods=student_mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)
student_model = student_model.to(device).train()

print("Teacher model loaded from:", teacher_ckpt_path)
print("Student model defined for modalities:", student_mods)

In [ ]:
# Run knowledge distillation
save_distilled_path = "/content/drive/MyDrive/hive_aud_distilled.pt" if os.path.exists("/content/drive") else "/content/hive_aud_distilled.pt"

distill_train(
    teacher=teacher_model,
    student=student_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100,
    lr=3e-4,
    T=2.0,
    w_hard=1.0,
    w_soft=0.5,
    w_reg=0.25,
    w_z=0.2,
    save_path=save_distilled_path
)

print("Distillation training complete. Distilled student model saved to:", save_distilled_path)

The knowledge distillation process aims to transfer the capabilities of a larger, more complex 'teacher' model (in this case, the `fused_all` multimodal model) to a smaller, more efficient 'student' model (here, an `aud_only` model). This is done by training the student not only on the ground-truth labels (hard labels) but also on the softened predictions and latent representations of the teacher model (soft labels and feature distillation).

After the distillation training, the `aud_only` student model should ideally achieve better performance than it would have if trained from scratch, especially in areas where the teacher had strong performance due to other modalities.

In [ ]:
print("DataFrame head:")
display(df.head())

print("DataFrame info:")
df.info()

In [ ]:
E_text = embed_text(texts, batch_size=32)
E_img  = embed_images(img_paths, batch_size=32)
E_aud  = embed_audio(aud_paths, batch_size=8)

print(E_text.shape, E_img.shape, E_aud.shape)

CACHE = DATA_ROOT / "emb_cache"
CACHE.mkdir(parents=True, exist_ok=True)

np.save(CACHE / "E_text.npy", E_text)
np.save(CACHE / "E_img.npy",  E_img)
np.save(CACHE / "E_aud.npy",  E_aud)
